In [ ]:
# %load_ext autoreload
# %autoreload 2

import sys
sys.path.append('/home/galk/LanguageDynamics/src') 
from data_generation import *
from config import TinyLMConfig, TinyDVAEConfig, TrainingConfig, TrainingDVAEConfig, TrainingDAVBConfig, ExperimentConfig, ExperimentAVBConfig
from models import TinyLlamaTransformer
from models import TransformerAutoencoder, TransformerDVAE, kl_divergence_gaussians
from models import TransformerDVAEFixedEncoder
from models import TinyLlamaCritic
import random
import numpy as np
from tqdm import tqdm


import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.distributions.kl import kl_divergence
from torch.distributions.multivariate_normal import MultivariateNormal
# from datasets import load_dataset
from datetime import datetime

from importlib import reload

models_path = '/home/galk/LanguageDynamics/models/linguistic_flip_flop'

In [ ]:
reload(sys.modules['models.transformers'])

In [ ]:
NT_vocab = ["E1", "D1", "M1", "M2"]
NT_token2id = {tok: idx for idx, tok in enumerate(NT_vocab)}
NT_id2token = {idx: tok for tok, idx in NT_token2id.items()}
NT_vocab_size = len(NT_vocab)

In [ ]:
### Hyperparameters - Original, non-symmetric noise task
E = 1
D = 1
N = 4
M = 2
G = 8
L_E = [1 for _ in range(E)]
L_D = [1 for _ in range(D)]
L_N = [3, 4, 5, 6]
L_M = [1 for _ in range(M)]

special_tokens = []

G_MIN = len(special_tokens) + E + D + M  # special tokens, plus other tokens
G_MAX = G_MIN + G - 1

memory_limit = float("inf")

### Sample noise trajectories
noise_list = []
for n_i in range(N):
    noise_list.append(random.sample(range(G_MIN, G_MAX+1), L_N[n_i]))

vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"G{g+1}" for g in range(G)]
token2id = {tok: idx for idx, tok in enumerate(vocab)}
id2token = {idx: tok for tok, idx in token2id.items()}
vocab_size = len(vocab)

# Non-Terminal (NT) vocab
NT_vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"N{n+1}" for n in range(N)]
NT_token2id = {tok: idx for idx, tok in enumerate(NT_vocab)}
NT_id2token = {idx: tok for tok, idx in NT_token2id.items()}
NT_vocab_size = len(NT_vocab)

# Initialize NT to T transitions
NT_to_T = {token: token if token not in [f"N{n+1}" for n in range(N)] else [id2token[id] for id in noise_list[[f"N{n+1}" for n in range(N)].index(token)]] for token in NT_vocab }

### Initialize Transition matrices
# P_transitions = np.zeros((E+1, NT_vocab_size, NT_vocab_size))  # [states, source, target]
P_transitions = np.zeros((1, NT_vocab_size, NT_vocab_size))  # [states, source, target]

# zero mode - no memory
# P_transitions[0, NT_token2id['<EOS>'], NT_token2id['<BOS>']] = 1
# P_transitions[0, NT_token2id.<BOS>'], NT_token2id.E1']] = 1/(2*N)
# P_transitions[0, NT_token2id['<BOS>'], NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/N
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(2*N)
P_transitions[0, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['<EOS>']] = 0.2
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 1/(N)
# P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['<EOS>']] = 1/(2*N)
np.fill_diagonal(P_transitions[0], 0)

# memory mode
# P_transitions[1, NT_token2id['D1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M  ## THIS WILL BE DECIDED BY THE CONTEXT (which M was memorized)
# P_transitions[1, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
# P_transitions[1, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 2/(N)
# np.fill_diagonal(P_transitions[1], 0)


# Normalize Matrices
P_transitions = P_transitions / P_transitions.sum(axis=-1, keepdims=True) # normalize rows to sum to 1
P_transitions = np.nan_to_num(P_transitions) # replace NaNs with 0


In [ ]:
### Hyperparameters - Simple Memory
E = 1
D = 1
N = 0
M = 2
G = 8
L_E = [1 for _ in range(E)]
L_D = [1 for _ in range(D)]
L_N = [3, 4, 5, 6]
L_M = [1 for _ in range(M)]

special_tokens = []

G_MIN = len(special_tokens) + E + D + M  # special tokens, plus other tokens
G_MAX = G_MIN + G - 1

memory_limit = float("inf")

### Sample noise trajectories
noise_list = []
for n_i in range(N):
    noise_list.append(random.sample(range(G_MIN, G_MAX+1), L_N[n_i]))

vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"G{g+1}" for g in range(G)]
token2id = {tok: idx for idx, tok in enumerate(vocab)}
id2token = {idx: tok for tok, idx in token2id.items()}
vocab_size = len(vocab)

# Non-Terminal (NT) vocab
NT_vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"N{n+1}" for n in range(N)]
NT_token2id = {tok: idx for idx, tok in enumerate(NT_vocab)}
NT_id2token = {idx: tok for tok, idx in NT_token2id.items()}
NT_vocab_size = len(NT_vocab)

# Initialize NT to T transitions
NT_to_T = {token: token if token not in [f"N{n+1}" for n in range(N)] else [id2token[id] for id in noise_list[[f"N{n+1}" for n in range(N)].index(token)]] for token in NT_vocab }

### Initialize Transition matrices
# P_transitions = np.zeros((E+1, NT_vocab_size, NT_vocab_size))  # [states, source, target]
P_transitions = np.zeros((1, NT_vocab_size, NT_vocab_size))  # [states, source, target]

# zero mode - no memory
# P_transitions[0, NT_token2id['<EOS>'], NT_token2id['<BOS>']] = 1
# P_transitions[0, NT_token2id.<BOS>'], NT_token2id.E1']] = 1/(2*N)
# P_transitions[0, NT_token2id['<BOS>'], NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/N
P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(2)
P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['D1']:NT_token2id[f'D{E}']+1] = 1/(2)
P_transitions[0, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['<EOS>']] = 0.2
# P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(N)
# P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 1/(N)
# P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['<EOS>']] = 1/(2*N)
np.fill_diagonal(P_transitions[0], 0)

# memory mode
# P_transitions[1, NT_token2id['D1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M  ## THIS WILL BE DECIDED BY THE CONTEXT (which M was memorized)
# P_transitions[1, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
# P_transitions[1, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 2/(N)
# np.fill_diagonal(P_transitions[1], 0)


# Normalize Matrices
P_transitions = P_transitions / P_transitions.sum(axis=-1, keepdims=True) # normalize rows to sum to 1
P_transitions = np.nan_to_num(P_transitions) # replace NaNs with 0


In [ ]:
### Hyperparameters - Symmetric Noise Task
E = 1
D = 1
N = 4
M = 2
G = 8
L_E = [1 for _ in range(E)]
L_D = [1 for _ in range(D)]
L_N = [3, 4, 5, 6]
L_M = [1 for _ in range(M)]

special_tokens = []

G_MIN = len(special_tokens) + E + D + M  # special tokens, plus other tokens
G_MAX = G_MIN + G - 1

memory_limit = float("inf")

### Sample noise trajectories
noise_list = []
for n_i in range(N):
    noise_list.append(random.sample(range(G_MIN, G_MAX+1), L_N[n_i]))

vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"G{g+1}" for g in range(G)]
token2id = {tok: idx for idx, tok in enumerate(vocab)}
id2token = {idx: tok for tok, idx in token2id.items()}
vocab_size = len(vocab)

# Non-Terminal (NT) vocab
NT_vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"N{n+1}" for n in range(N)]
NT_token2id = {tok: idx for idx, tok in enumerate(NT_vocab)}
NT_id2token = {idx: tok for tok, idx in NT_token2id.items()}
NT_vocab_size = len(NT_vocab)

# Initialize NT to T transitions
NT_to_T = {token: token if token not in [f"N{n+1}" for n in range(N)] else [id2token[id] for id in noise_list[[f"N{n+1}" for n in range(N)].index(token)]] for token in NT_vocab }

### Initialize Transition matrices
# P_transitions = np.zeros((E+1, NT_vocab_size, NT_vocab_size))  # [states, source, target]
P_transitions = np.zeros((1, NT_vocab_size, NT_vocab_size))  # [states, source, target]

# zero mode - no memory
# P_transitions[0, NT_token2id['<EOS>'], NT_token2id['<BOS>']] = 1
# P_transitions[0, NT_token2id.<BOS>'], NT_token2id.E1']] = 1/(2*N)
# P_transitions[0, NT_token2id['<BOS>'], NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/N
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(2*N)
P_transitions[0, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(N)
P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['D1']:NT_token2id[f'D{E}']+1] = 1/(N)
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['<EOS>']] = 0.2
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 1/(N)
# P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['<EOS>']] = 1/(2*N)
# np.fill_diagonal(P_transitions[0], 0)

# memory mode
# P_transitions[1, NT_token2id['D1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M  ## THIS WILL BE DECIDED BY THE CONTEXT (which M was memorized)
# P_transitions[1, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
# P_transitions[1, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 2/(N)
# np.fill_diagonal(P_transitions[1], 0)


# Normalize Matrices
P_transitions = P_transitions / P_transitions.sum(axis=-1, keepdims=True) # normalize rows to sum to 1
P_transitions = np.nan_to_num(P_transitions) # replace NaNs with 0


In [ ]:
model_config = TinyLMConfig(
    n_layers=4,
    n_heads=4,
    embed_dim=32,
    ffn_dim=256,
    context_window=32,
    vocab=NT_vocab,
    dropout_self_attention=0.05,
    dropout_embed=0.03,
    dropout_residual=0.03
)

training_config = TrainingConfig(
    lr=1e-4,
    batch_size=256,
    grad_clipping=True
)

experiment_config = ExperimentConfig(
    epochs=100,
    # checkpoint_path=None,
    # checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_LM_dyck2_layers_4_embed_32_ffn_dim_256_context_window_32_date_110925_1816_trial_1/ckpt_epoch2.pt",
    save_every=1,
    device_index=0,
    model_config=model_config,
    training_config=training_config
)

pad_id = 100

# TODO: move the model save prefix to the ExperimentConfig initialization
current_date_ddmmyy = datetime.now().strftime('%d%m%y_%H%M')
current_date_only_day = datetime.now().strftime('%d%m%y')
if model_config.mode == 'AE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
if model_config.mode == 'KAE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_n_diagonals_{model_config.n_diagonals}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
elif model_config.mode == 'LM' or model_config.mode == 'RLM':
    # TODO: add memory generation parameters to name
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_date_{current_date_ddmmyy}"

experiment_config.model_save_prefix = os.path.join(current_date_only_day, experiment_config.model_save_prefix)
print(f"Using device: {experiment_config.device}")
print(f"Model save location: {experiment_config.model_save_prefix}")

In [ ]:
context_window = experiment_config.model_config.context_window
n_train = 1
n_train_windows = 100000
max_steps = n_train_windows * (context_window + 1)
val_ratio = 0.01
train_dataset, train_dataset_NT, train_seen = generate_tiny_memory_dataset(n_samples=n_train, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=max_steps, memory_limit=memory_limit)
val_dataset, val_dataset_NT, val_seen = generate_tiny_memory_dataset(n_samples=n_train, seen=train_seen, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=int(max_steps*val_ratio), memory_limit=memory_limit)
train_blocks = prepare_blocks(train_dataset_NT, NT_token2id, context_window)
val_blocks = prepare_blocks(val_dataset_NT, NT_token2id, context_window)

In [ ]:
print(train_blocks.shape)
print(val_blocks.shape)

In [ ]:
# Create datasets
train_dataset = TokensDataset(train_blocks)
val_dataset = TokensDataset(val_blocks)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=experiment_config.training_config.batch_size,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=experiment_config.training_config.batch_size,
    shuffle=False,  # No need to shuffle validation data
    drop_last=True
)

# Verify the split
print("\nDataLoader Info:")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# --------- 7. Training --------- WITH CHECKPOINT LOADING 26/06/25
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        self.epochs = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_losses = metrics_dict['train_losses']
        self.train_accuracies = metrics_dict['train_accuracies']
        self.val_losses = metrics_dict['val_losses']
        self.val_accuracies = metrics_dict['val_accuracies']
        self.epochs = list(range(1, len(self.train_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

        # Plot losses
        ax1.plot(self.epochs, self.train_losses, label='Train Loss')
        ax1.plot(self.epochs, self.val_losses, label='Val Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Training and Validation Loss')
        ax1.legend()
        ax1.grid(True)

        # Plot accuracies
        ax2.plot(self.epochs, self.train_accuracies, label='Train Accuracy')
        ax2.plot(self.epochs, self.val_accuracies, label='Val Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.set_title('Training and Validation Accuracy')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png")

        plt.show()

def load_checkpoint(checkpoint_path, model, device, optimizer=None):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch and loaded metrics
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    for key in ['vocab_size', 'embed_dim', 'n_layers', 'n_heads', 'ffn_dim', 'context_window']:
        if CONFIG[key] != saved_config[key]:
            raise ValueError(f"Checkpoint config mismatch for {key}: "
                           f"current={CONFIG[key]}, saved={saved_config[key]}")

    starting_epoch = checkpoint['epoch']
    return starting_epoch, checkpoint['metrics']

n_trials = 1
device = experiment_config.device
for trial in range(n_trials):
    # Initialize model
    if experiment_config.model_config.mode == 'LM':
        model = TinyLlamaTransformer(
            experiment_config.model_config
        ).to(device)

    elif experiment_config.model_config.mode == 'AE':
        model = TransformerAutoencoder(
            experiment_config.model_config
        ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=experiment_config.training_config.lr)
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, experiment_config.model_save_prefix + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    if experiment_config.checkpoint_path:
        try:
            starting_epoch, saved_metrics = load_checkpoint(
                experiment_config.checkpoint_path,
                model,
                device,
                optimizer
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = experiment_config.training_config.lr
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0

    print("Starting training loop...")
    global_start = time.time()

    def validate(model, val_loader, criterion, model_config, device):
        model.eval()
        total_loss = 0
        total_acc = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                if model_config.mode == 'LM':
                    logits = model(x)
                elif model_config == 'AE':
                    logits, _, _, _ = model(x, decoding_seed=x[:, :-1])
                loss = criterion(logits.view(-1, len(model_config.vocab)), y.view(-1))
                acc = calculate_accuracy(logits, y, pad_id)
                total_loss += loss.item()
                total_acc += acc
        return total_loss / len(val_loader), total_acc / len(val_loader)

    # Training loop
    for epoch in range(starting_epoch, experiment_config.epochs):
        model.train()
        epoch_loss = 0.0
        epoch_acc = 0.0
        start = time.time()

        # Training phase
        for batch, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            x = x.to(device)
            y = y.to(device)

            if experiment_config.model_config.mode == 'LM':
                logits = model(x)
            elif experiment_config.model_config.mode == 'AE':
                logits, _, _, _ = model(x, decoding_seed=x[:,:-1])
            loss = criterion(logits.view(-1, len(experiment_config.model_config.vocab)), y.view(-1))
            acc = calculate_accuracy(logits, y, pad_id)

            optimizer.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            if experiment_config.training_config.grad_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            epoch_loss += loss.item()
            epoch_acc += acc

        # Calculate training metrics
        train_loss = epoch_loss / len(train_loader)
        train_acc = epoch_acc / len(train_loader)

        # Validation phase
        val_loss, val_acc = validate(model, val_loader, criterion, experiment_config.model_config, device)

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_losses.append(train_loss)
        metrics.train_accuracies.append(train_acc)
        metrics.val_losses.append(val_loss)
        metrics.val_accuracies.append(val_acc)

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")

        # Plot metrics
        metrics.plot_metrics(save_dir)

        # Save checkpoint
        if (epoch+1) % experiment_config.save_every == 0 or (epoch+1) == experiment_config.epochs:
            # ckpt_path = save_dir / f"{CONFIG['model_save_prefix']}_trial_{trial+1}_epoch{epoch+1}.pt"
            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': experiment_config,
                'epoch': epoch+1,
                'metrics': {
                    'train_losses': metrics.train_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'val_losses': metrics.val_losses,
                    'val_accuracies': metrics.val_accuracies
                }
            }, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    print("Training complete!")

In [ ]:
# Loading a saved model
# ckpt_path = experiment_config.checkpoint_path
# ckpt_path = "/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_LM_tinymemory_layers_2_embed_8_ffn_dim_128_context_window_32_date_211025_1759_trial_1/ckpt_epoch15.pt"
ckpt_path = "/home/galk/LanguageDynamics/models/linguistic_flip_flop/291025/tiny_LM_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_date_291025_1514_trial_1/ckpt_epoch9.pt" # new symmetric noise
checkpoint = torch.load(ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)

# if 'mode' not in checkpoint['config']:
    # checkpoint['config']['mode'] = "LM"  # Default to LM if not specified

if checkpoint['config'].model_config.mode == 'LM':
    model = TinyLlamaTransformer(
        checkpoint['config'].model_config
    ).to(experiment_config.device)

elif checkpoint['config'].model_config.mode == 'RLM':
        model = TinyLlamaRawTransformer(
            embed_dim=CONFIG['embed_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window']
        ).to(CONFIG['device'])

elif checkpoint['config'].model_config.mode == 'AE':
    model = TransformerAutoencoder(
            vocab_size=checkpoint['config']['vocab_size'],
            embed_dim=checkpoint['config']['embed_dim'],
            latent_dim=checkpoint['config']['latent_dim'],
            n_layers=checkpoint['config']['n_layers'],
            n_heads=checkpoint['config']['n_heads'],
            ffn_dim=checkpoint['config']['ffn_dim'],
            context_window=checkpoint['config']['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=checkpoint['config']['n_latents']
        ).to(CONFIG['device'])
elif checkpoint['config'].model_config.mode == 'KAE':
    model = TransformerKoopmanAutoencoder(
            vocab_size=CONFIG['vocab_size'],
            embed_dim=CONFIG['embed_dim'],
            latent_dim=CONFIG['latent_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=CONFIG['n_latents'],
            n_diagonals=CONFIG['n_diagonals']
        ).to(CONFIG['device'])


model_state = checkpoint['model_state_dict']
model.load_state_dict(model_state)

In [ ]:
prompt = ['E1', "M2"]

max_new_tokens = 80
temperature = 1
top_k = 10

generated, probs = generate_from_model(
    model=model,
    token2id=NT_token2id,
    id2token=NT_id2token,
    prompt=prompt,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_k=top_k,
    device=experiment_config.device
)

print(generated)

In [ ]:

fig = plot_generation_probabilities(
    probabilities=probs,
    sequence=generated[len(prompt):],
    token2id=NT_token2id,
    id2token=NT_id2token,
    tokens_to_highlight=None,
    figsize=(12, 8)
)

In [ ]:
# generate trajectories for DVAE training

context_window = experiment_config.model_config.context_window
n_train = 500
n_train_windows = 1
max_steps = n_train_windows * (context_window + 1)
n_val = 80
train_dataset, train_dataset_NT, train_seen = generate_tiny_memory_dataset(n_samples=n_train, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=max_steps, memory_limit=memory_limit)
val_dataset, val_dataset_NT, val_seen = generate_tiny_memory_dataset(n_samples=n_val, seen=train_seen, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=max_steps, memory_limit=memory_limit)
# train_blocks = prepare_blocks(train_dataset_NT, NT_token2id, context_window)
# val_blocks = prepare_blocks(val_dataset_NT, NT_token2id, context_window)

max_new_tokens = 50
temperature = 1
top_k = 10
trajectories_per_initial_seq = 5

# returns a np.array of encoded trajectories, shape [B, T]
train_trajectories = create_stacked_trajectories_array(
    initial_seqs=train_dataset_NT,
    context_window=context_window,
    trajectories_per_initial_seq=trajectories_per_initial_seq,
    model=model,
    token2id=NT_token2id,
    id2token=NT_id2token,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_k=top_k,
    device=experiment_config.device
)

val_trajectories = create_stacked_trajectories_array(
    initial_seqs=val_dataset_NT,
    context_window=context_window,
    trajectories_per_initial_seq=trajectories_per_initial_seq,
    model=model,
    token2id=NT_token2id,
    id2token=NT_id2token,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_k=top_k,
    device=experiment_config.device
)

In [ ]:
# print(train_trajectories[0])
# print(train_trajectories[1])
# print([NT_id2token[id] for id in train_trajectories[0]])
# print([NT_id2token[id] for id in train_trajectories[1]])

In [ ]:
# Save trajectories
np.save('/home/galk/LanguageDynamics/data/tiny_memory_train_context_window_32_num_steps_50_2500_tokenized_trajectories_flat_311025_4_layers_LLM_symmetric_noise.npy', train_trajectories)
np.save('/home/galk/LanguageDynamics/data/tiny_memory_val_context_window_32_num_steps_50_400_tokenized_trajectories_flat_311025_4_layers_LLM_symmetric_noise.npy', val_trajectories)

In [ ]:
# Load trajectories
# Original Flip Flop task - with non-equivalent nose
# train_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_train_context_window_32_num_steps_50_1000_tokenized_trajectories_flat.npy')
# val_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_val_context_window_32_num_steps_50_250_tokenized_trajectories_flat.npy')

# Simple memory task (no noise) - first attempt
# train_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_train_context_window_32_num_steps_50_1000_tokenized_trajectories_flat_FOR_FIXED_ENCODER.npy')
# val_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_val_context_window_32_num_steps_50_250_tokenized_trajectories_flat_FOR_FIXED_ENCODER.npy')

# Simple Memory Task (no noise) - new after fixed LM loading
# train_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_train_context_window_32_num_steps_50_1000_tokenized_trajectories_flat_221025_4_layers_LLM.npy')
# val_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_val_context_window_32_num_steps_50_250_tokenized_trajectories_flat_221025_4_layers_LLM.npy')

# Symmetric Noise Task - new after fixed LM loading
train_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_train_context_window_32_num_steps_50_2500_tokenized_trajectories_flat_311025_4_layers_LLM_symmetric_noise.npy')
val_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_val_context_window_32_num_steps_50_400_tokenized_trajectories_flat_311025_4_layers_LLM_symmetric_noise.npy')

In [ ]:
# DVAE config
model_config = TinyDVAEConfig(
    n_layers=4,
    n_heads=4,
    embed_dim=32,
    ffn_dim=256,
    context_window=32,
    vocab=NT_vocab,
    dropout_self_attention=0.05,
    dropout_embed=0.03,
    dropout_residual=0.03,
    dropout_latent=0.00,
    latent_dim=3,
    decoder_ffn_dim=64,
    dropout_decoder=0.03,
    transition_ffn_dim=64,
    dropout_transition=0.03,
    # pooling='last',
    # pooling='none',
    pooling='attention',
    decoder_ln=False,
    transition_ln=False
)

training_config = TrainingDVAEConfig(
    lr=5e-4,
    batch_size=64,
    grad_clipping=True,
    reconstruction_coef=1.0,
    warmup_steps=1500,
    minimal_beta=0.00,
    maximal_beta=1.0,
    n_encoder_layers=4,
    teacher_forcing=True,
    use_scheduler=True,
    scheduler_T_max=300,
    scheduler_eta_min=1e-5
)

experiment_config = ExperimentConfig(
    epochs=500,
    # checkpoint_path=None,
    checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/311025/tiny_DVAE_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_latent_3_date_311025_1351_trial_1/ckpt_epoch50.pt",
    save_every=10,
    device_index=0,
    model_config=model_config,
    training_config=training_config,
    # LM_checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_LM_dyck2_layers_4_embed_32_ffn_dim_256_context_window_32_date_110925_1816_trial_1/ckpt_epoch2.pt"
    # LM_checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_LM_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_date_121025_1841_trial_1/ckpt_epoch2.pt"
    # LM_checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/221025/tiny_LM_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_date_221025_1107_trial_1/ckpt_epoch5.pt" # new simple memory (no noise) LM after fixed loading
    LM_checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/291025/tiny_LM_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_date_291025_1514_trial_1/ckpt_epoch9.pt" # new symmetric noise
)

pad_id = 100

# TODO: move the model save prefix to the ExperimentConfig initialization
current_date_ddmmyy = datetime.now().strftime('%d%m%y_%H%M')
current_date_only_day = datetime.now().strftime('%d%m%y')
if model_config.mode == 'AE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
if model_config.mode == 'KAE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_n_diagonals_{model_config.n_diagonals}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
if model_config.mode == 'DVAE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_date_{current_date_ddmmyy}"
elif model_config.mode == 'LM' or model_config.mode == 'RLM':
    # TODO: add memory generation parameters to name
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_date_{current_date_ddmmyy}"

experiment_config.model_save_prefix = os.path.join(current_date_only_day, experiment_config.model_save_prefix)
print(f"Using device: {experiment_config.device}")
print(f"Model save location: {experiment_config.model_save_prefix}")

In [ ]:
teacher_model = TransformerDVAE(
            experiment_config.model_config
        ).to(experiment_config.device)

# ckpt_path = experiment_config.checkpoint_path
ckpt_path = "/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_DVAE_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_latent_2_date_191025_1752_trial_1/ckpt_epoch200_FIXED_TEACHER.pt"
checkpoint = torch.load(ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)

teacher_model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
train_trajectory_dataset = TrajectoryDataset(train_trajectories)
val_trajectory_dataset = TrajectoryDataset(val_trajectories)

In [ ]:
def collate_fn_np(batch):
    # 'batch' is a list of np arrays, of shape [T] of trajectory IDs
    trajs = np.array(batch) # [B, T]
    trajs = torch.tensor(trajs, dtype=torch.long)  # shape [B, T]
    return trajs

train_loader = DataLoader(
    train_trajectory_dataset, 
    batch_size=experiment_config.training_config.batch_size, 
    shuffle=True, 
    collate_fn=collate_fn_np, 
    drop_last=True,
    # num_workers=16,
    # prefetch_factor=2,
    # persistent_workers=True,
    # pin_memory=True
    )

val_loader = DataLoader(
    val_trajectory_dataset, 
    # batch_size=experiment_config.training_config.batch_size, 
    batch_size=experiment_config.training_config.batch_size,
    shuffle=False, 
    collate_fn=collate_fn_np, 
    drop_last=True,
    # num_workers=16,
    # prefetch_factor=2,
    # persistent_workers=True,
    # pin_memory=True
    )


In [ ]:
### Dynamical VAE Training
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        # Training metrics
        self.train_total_losses = []
        self.train_reconstruction_losses = []
        self.train_kl_losses = []
        self.train_accuracies = []
        
        # Validation metrics
        self.val_reconstruction_losses = []
        self.val_kl_losses = []
        self.val_accuracies = []
        
        self.epochs = []
        self.learning_rates = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_total_losses = metrics_dict.get('train_total_losses', [])
        self.train_reconstruction_losses = metrics_dict.get('train_reconstruction_losses', [])
        self.train_kl_losses = metrics_dict.get('train_kl_losses', [])
        self.train_accuracies = metrics_dict.get('train_accuracies', [])
        
        self.val_reconstruction_losses = metrics_dict.get('val_reconstruction_losses', [])
        self.val_kl_losses = metrics_dict.get('val_kl_losses', [])
        self.val_accuracies = metrics_dict.get('val_accuracies', [])
        self.learning_rates = metrics_dict.get('learning_rates', [])
        
        self.epochs = list(range(1, len(self.train_total_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Color scheme
        train_color = "#287BBB"  # Blue
        val_color = "#ED9E00"    # Orange
        
        # Plot 1: Reconstruction losses
        axes[0,0].plot(self.epochs, self.train_reconstruction_losses, 
                    color=train_color, linestyle='-', label='Train Reconstruction')
        axes[0,0].plot(self.epochs, self.val_reconstruction_losses, 
                    color=val_color, linestyle='-', label='Val Reconstruction')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Loss')
        axes[0,0].set_title('Reconstruction Loss')
        axes[0,0].legend()
        axes[0,0].grid(True)

        # Plot 2: KL losses
        axes[0,1].plot(self.epochs, self.train_kl_losses, 
                    color=train_color, linestyle='-', label='Train KL')
        axes[0,1].plot(self.epochs, self.val_kl_losses, 
                    color=val_color, linestyle='-', label='Val KL')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Loss')
        axes[0,1].set_title('KL Divergence Loss')
        axes[0,1].legend()
        axes[0,1].grid(True)

        # Plot 3: Total training loss and LR
        ax3_twin = axes[1,0].twinx()
        axes[1,0].plot(self.epochs, self.train_total_losses, color=train_color, label='Total Train Loss')
        ax3_twin.plot(self.epochs, self.learning_rates, color='green', linestyle='--', label='Learning Rate')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Loss')
        ax3_twin.set_ylabel('Learning Rate')
        axes[1,0].set_title('Total Training Loss and Learning Rate')
        axes[1,0].legend(loc='upper left')
        ax3_twin.legend(loc='upper right')
        axes[1,0].grid(True)


        # Plot 4: Accuracies
        axes[1,1].plot(self.epochs, self.train_accuracies, 
                    color=train_color, linestyle='-', label='Train Accuracy')
        axes[1,1].plot(self.epochs, self.val_accuracies, 
                    color=val_color, linestyle='-', label='Val Accuracy')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Accuracy')
        axes[1,1].set_title('Model Accuracy')
        axes[1,1].legend()
        axes[1,1].grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png", dpi=300, bbox_inches='tight')

        plt.show()

def load_checkpoint(checkpoint_path, model, optimizer, scheduler, experiment_config, device):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch, loaded metrics, and global step
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler state if provided
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    # Compare relevant parts of the model config
    current_model_config = experiment_config.model_config
    saved_model_config = saved_config.model_config
    
    mismatched_keys = []
    for key in ['n_layers', 'n_heads', 'embed_dim', 'ffn_dim', 'context_window', 'latent_dim']:
        if getattr(current_model_config, key) != getattr(saved_model_config, key):
            mismatched_keys.append(key)
    
    if mismatched_keys:
        raise ValueError(f"Checkpoint config mismatch for keys: {mismatched_keys}")

    starting_epoch = checkpoint['epoch']
    global_step = checkpoint.get('global_step', 0)
    return starting_epoch, checkpoint['metrics'], global_step

n_trials = 1
for trial in range(n_trials):
    # Initialize model
    if experiment_config.model_config.mode == 'DVAE':
        model = TransformerDVAE(
            experiment_config.model_config
        ).to(experiment_config.device)

    else:
        raise ValueError(f"Unsupported mode: {experiment_config.model_config.mode}")

    ### LOAD TRAINED ENCODER
    # with torch.no_grad():
    #     model.to_mu.weight.copy_(teacher_model.to_mu.weight)
    #     model.to_mu.bias.copy_(teacher_model.to_mu.bias)
    #     model.to_logvar.weight.copy_(teacher_model.to_logvar.weight)
    #     model.to_logvar.bias.copy_(teacher_model.to_logvar.bias)
    #     model.encoder_ln.weight.copy_(teacher_model.encoder_ln.weight)
    #     model.encoder_ln.bias.copy_(teacher_model.encoder_ln.bias)

    # Load a pretrained LM model to use its encoder
    # Loading a saved model
    lm_ckpt_path = experiment_config.LM_checkpoint_path
    # ckpt_path = "/content/tiny_transformer_dyck2_layers_2_embed_32_context_window_64_weight_tied_maxdepth_20_max_len_50_date_29_07_25_trial_1/model_epoch15.pt"
    if lm_ckpt_path:
        LM_checkpoint = torch.load(lm_ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)

        if LM_checkpoint['config'].model_config.mode == 'LM':
            lm_model = TinyLlamaTransformer(
                LM_checkpoint['config'].model_config
            ).to(experiment_config.device)
        lm_model.load_state_dict(LM_checkpoint['model_state_dict']) ## IMPORTANT!!!
        model.encoder = lm_model
        for param in model.encoder.parameters():  # freeze the encoder
            param.requires_grad = False

    params_to_train = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = torch.optim.AdamW(params_to_train, lr=experiment_config.training_config.lr)
    scheduler = None
    if experiment_config.training_config.use_scheduler:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=experiment_config.training_config.scheduler_T_max,
            eta_min=experiment_config.training_config.scheduler_eta_min
        )
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, experiment_config.model_save_prefix + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    global_step = 0
    if experiment_config.checkpoint_path:
        try:
            starting_epoch, saved_metrics, global_step = load_checkpoint(
                experiment_config.checkpoint_path,
                model,
                optimizer,
                scheduler,
                experiment_config,
                experiment_config.device
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = experiment_config.training_config.lr
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}, global_step {global_step}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0
            global_step = 0

    print("Starting training loop...")
    global_start = time.time()


    def validate(model: TransformerDVAE, val_loader, criterion, device):
        model.eval()
        total_reconstruction_loss = 0
        total_kl_loss = 0
        total_acc = 0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device)
                
                # x is [B, T] tensor of token trajectories. Virtually stack it into [B, T', C] windows
                B, T_data = x.shape
                C = model.context_window
                x_unfolded = x.unfold(dimension=-1, size=C, step=1)
                B, T, C_out = x_unfolded.shape
                
                x_reshaped = x_unfolded.reshape(B * T, C_out)
                
                # inference: encode observations into latents
                z, mu_q, logvar_q = model.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers) # z, mu, logvar: [B*T, latent_dim]

                # decode the latents to reconstruct the observations
                logits = model.decode(z) # logits: [B*T, vocab_size]

                loss_reconstruction = criterion(logits, x_reshaped[:, -1])
                acc = calculate_accuracy(logits, x_reshaped[:, -1], pad_id)

                # reshape z to [B, T, latent_dim] for KL calculation
                z = z.reshape(B, T, model.latent_dim)
                
                # initial_latent = torch.zeros((B, model.latent_dim), device=device)
                # mu_p_t0, logvar_p_t0 = model.transition_model(initial_latent, is_t0=True)  # [B, latent_dim]
                # z_prev = torch.cat([initial_latent.unsqueeze(1), z[:, :-1, :]], dim=1)  # [B, T, latent_dim]
                z_prev = z[:, :-1, :]  # [B, T-1, latent_dim]
                mu_p, logvar_p = model.transition_model(z_prev.reshape(B*(T-1), -1))  # [B*(T-1), latent_dim]

                # # concatenate t=0 prior
                # mu_p = mu_p.reshape(B, T-1, -1)
                # logvar_p = logvar_p.reshape(B, T-1, -1)
                # mu_p = torch.cat([mu_p_t0.unsqueeze(1), mu_p], dim=1).reshape(B*T, -1)
                # logvar_p = torch.cat([logvar_p_t0.unsqueeze(1), logvar_p], dim=1).reshape(B*T, -1)

                # remove first posterior from each trajectory (it cannot be predicted) to match prior shape
                mu_q = mu_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)
                logvar_q = logvar_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)

                # # construct L matrix for transition priors
                # var_p_log_diag = logvar_p[:, :model.latent_dim]
                # var_p_off_diag = logvar_p[:, model.latent_dim:]
                # positive_diag = torch.exp(var_p_log_diag)
                # L = torch.zeros(logvar_p.shape[0], model.latent_dim, model.latent_dim, device=logvar_q.device)
                # tril_indices = torch.tril_indices(row=model.latent_dim, col=model.latent_dim, offset=-1)
                # L[:, tril_indices[0], tril_indices[1]] = var_p_off_diag
                # L += torch.diag_embed(positive_diag)

                L_q = model.build_cholesky_L(logvar_q)
                L_p = model.build_cholesky_L(logvar_p)

                # dist_q = MultivariateNormal(loc=mu_q, covariance_matrix=torch.diag_embed(torch.exp(logvar_q)))
                dist_q = MultivariateNormal(loc=mu_q, scale_tril=L_q)
                # dist_p = MultivariateNormal(loc=mu_p, covariance_matrix=torch.diag_embed(torch.exp(logvar_p)))
                dist_p = MultivariateNormal(loc=mu_p, scale_tril=L_p)

                KL_dist = kl_divergence(dist_q, dist_p)

                # Compute KL divergence loss
                # Dkl = kl_divergence_gaussians(mu_q, logvar_q, mu_p, logvar_p)  # [B*(T-1)]
                loss_kl = KL_dist.sum() / (B * (T-1) * model.latent_dim)

                total_reconstruction_loss += loss_reconstruction.item()
                total_kl_loss += loss_kl.item()
                total_acc += acc

                # predictions distributional statistics
                mean_mu_q = mu_q.mean(dim=0).detach().cpu()
                std_mu_q = mu_q.std(dim=0).detach().cpu()
                mean_logvar_q = logvar_q.mean(dim=0).detach().cpu()
                std_logvar_q = logvar_q.std(dim=0).detach().cpu()
                mean_mu_p = mu_p.mean(dim=0).detach().cpu()
                std_mu_p = mu_p.std(dim=0).detach().cpu()
                mean_logvar_p = logvar_p.mean(dim=0).detach().cpu()
                std_logvar_p = logvar_p.std(dim=0).detach().cpu()

                correlation_matrix = torch.corrcoef(torch.stack((mean_mu_q, mean_mu_p)))

                # The Pearson correlation coefficient between x and y is at index [0, 1] or [1, 0]
                pearson_r = correlation_matrix[0, 1]

                # print(f"val batch:")
                # print(f"mean_mu_q={mean_mu_q}, \nstd_mu_q={std_mu_q}, \nmean_logvar_q={mean_logvar_q}, \nstd_logvar_q={std_logvar_q}")
                # print(f"mean_mu_p={mean_mu_p}, \nstd_mu_p={std_mu_p}, \nmean_logvar_p={mean_logvar_p}, \nstd_logvar_p={std_logvar_p}")
                # print(f"mean_mu correlation: {pearson_r}")

        return total_reconstruction_loss / len(val_loader), total_kl_loss / len(val_loader), total_acc / len(val_loader)

    # Training loop
    device = experiment_config.device
    for epoch in range(starting_epoch, experiment_config.epochs):
        model.train()
        if lm_ckpt_path:
            model.encoder.eval()  # keep LM encoder in eval mode
        epoch_loss = 0.0
        epoch_loss_reconstruction = 0.0
        epoch_loss_kl = 0.0
        epoch_acc = 0.0
        start = time.time()

        # Training phase
        for batch, x in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            x = x.to(device)
            
            # x is [B, T] tensor of token trajectories. Virtually stack it into [B, T', C] windows
            B, T_data = x.shape
            C = model.context_window
            x_unfolded = x.unfold(dimension=-1, size=C, step=1)
            B, T, C_out = x_unfolded.shape
            
            x_reshaped = x_unfolded.reshape(B * T, C_out)
            
            # inference: encode observations into latents
            z, mu_q, logvar_q = model.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers) # z, mu, logvar: [B*T, latent_dim]

            # decode the latents to reconstruct the observations
            logits = model.decode(z) # logits: [B*T, vocab_size]

            loss_reconstruction = criterion(logits, x_reshaped[:, -1]) # reconstruction loss only for the last token in each window
            acc = calculate_accuracy(logits, x_reshaped[:, -1], pad_id)  # autoencoding reconstruction accuracy

            # reshape z to [B, T, latent_dim] for KL calculation
            z = z.reshape(B, T, model.latent_dim)

            # # TODO: add option to sample from initial prior instead of zero
            # initial_latent = torch.zeros((B, model.latent_dim), device=device)
            # mu_p_t0, logvar_p_t0 = model.transition_model(initial_latent, is_t0=True)  # [B, latent_dim]
            # z_prev = torch.cat([initial_latent.unsqueeze(1), z[:, :-1, :]], dim=1)  # [B, T, latent_dim]
            z_prev = z[:, :-1, :]  # [B, T-1, latent_dim]
            mu_p, logvar_p = model.transition_model(z_prev.reshape(B*(T-1), -1))  # [B*(T-1), latent_dim]

            # # concatenate t=0 prior
            # mu_p = mu_p.reshape(B, T-1, -1)
            # logvar_p = logvar_p.reshape(B, T-1, -1)
            # mu_p = torch.cat([mu_p_t0.unsqueeze(1), mu_p], dim=1).reshape(B*T, -1)
            # logvar_p = torch.cat([logvar_p_t0.unsqueeze(1), logvar_p], dim=1).reshape(B*T, -1)

            # remove first posterior from each trajectory (it cannot be predicted) to match prior shape
            mu_q = mu_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)
            logvar_q = logvar_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)

            # # construct L matrix for transition priors
            # var_p_log_diag = logvar_p[:, :model.latent_dim]
            # var_p_off_diag = logvar_p[:, model.latent_dim:]
            # positive_diag = torch.exp(var_p_log_diag)
            # L = torch.zeros(logvar_p.shape[0], model.latent_dim, model.latent_dim, device=logvar_q.device)
            # tril_indices = torch.tril_indices(row=model.latent_dim, col=model.latent_dim, offset=-1)
            # L[:, tril_indices[0], tril_indices[1]] = var_p_off_diag
            # L += torch.diag_embed(positive_diag)

            L_q = model.build_cholesky_L(logvar_q)
            L_p = model.build_cholesky_L(logvar_p)

            # dist_q = MultivariateNormal(loc=mu_q, covariance_matrix=torch.diag_embed(torch.exp(logvar_q)))
            dist_q = MultivariateNormal(loc=mu_q, scale_tril=L_q)
            # dist_p = MultivariateNormal(loc=mu_p, covariance_matrix=torch.diag_embed(torch.exp(logvar_p)))
            dist_p = MultivariateNormal(loc=mu_p, scale_tril=L_p)

            KL_dist = kl_divergence(dist_q, dist_p)

            # Compute KL divergence loss
            # Dkl = kl_divergence_gaussians(mu_q, logvar_q, mu_p, logvar_p)  # [B*(T-1)]
            loss_kl = KL_dist.sum() / (B * (T-1) * model.latent_dim)

            # print(f"torch.kl: {KL_dist}, my KL: {Dkl}, KL loss: {loss_kl}")

            # compute KL-annealing beta
            beta = (experiment_config.training_config.maximal_beta - experiment_config.training_config.minimal_beta) * min(1.0, global_step / experiment_config.training_config.warmup_steps) + experiment_config.training_config.minimal_beta
            loss = experiment_config.training_config.reconstruction_coef * loss_reconstruction + beta * loss_kl
            # loss = beta * loss_reconstruction + experiment_config.training_config.reconstruction_coef * loss_kl

            optimizer.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            if experiment_config.training_config.grad_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            global_step += 1

            epoch_loss += loss.item()
            epoch_loss_reconstruction += loss_reconstruction.item()
            epoch_loss_kl += loss_kl.item()
            epoch_acc += acc

        # Validation phase
        val_reconstruction_loss, val_kl_loss, val_acc = validate(model, val_loader, criterion, device)

        # Calculate average losses and accuracies
        train_loss = epoch_loss / len(train_loader)
        train_acc = epoch_acc / len(train_loader)
        train_loss_reconstruction = epoch_loss_reconstruction / len(train_loader)
        train_loss_kl = epoch_loss_kl / len(train_loader)

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_total_losses.append(train_loss)
        metrics.train_reconstruction_losses.append(train_loss_reconstruction)
        metrics.train_kl_losses.append(train_loss_kl)
        metrics.train_accuracies.append(train_acc)
        metrics.val_reconstruction_losses.append(val_reconstruction_loss)
        metrics.val_kl_losses.append(val_kl_loss)
        metrics.val_accuracies.append(val_acc)
        metrics.learning_rates.append(optimizer.param_groups[0]['lr'])

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Total Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Train Reconstruction Loss: {train_loss_reconstruction:.4f} | Train KL Loss: {train_loss_kl:.4f}")
        print(f"Val Reconstruction Loss: {val_reconstruction_loss:.4f} | Val KL Loss: {val_kl_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")
        
        # Step the scheduler
        if scheduler:
            scheduler.step()
            print(f"LR: {scheduler.get_last_lr()[0]:.6f}")

        # Save checkpoint
        if (epoch+1) % experiment_config.save_every == 0 or (epoch+1) == experiment_config.epochs:
            # Plot metrics
            metrics.plot_metrics(save_dir)

            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            checkpoint_data = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': experiment_config,
                'epoch': epoch+1,
                'global_step': global_step,
                'metrics': {
                    'train_total_losses': metrics.train_total_losses,
                    'train_reconstruction_losses': metrics.train_reconstruction_losses,
                    'train_kl_losses': metrics.train_kl_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'val_reconstruction_losses': metrics.val_reconstruction_losses,
                    'val_kl_losses': metrics.val_kl_losses,
                    'val_accuracies': metrics.val_accuracies,
                    'learning_rates': metrics.learning_rates
                }
            }
            if scheduler:
                checkpoint_data['scheduler_state_dict'] = scheduler.state_dict()
            
            torch.save(checkpoint_data, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")


    print("Training complete!")

In [ ]:
# checkpoint_path = "/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_DVAE_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_latent_5_date_061025_1844_trial_1/ckpt_epoch470.pt"
# checkpoint_path = "/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_DVAE_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_latent_2_date_191025_1851_trial_1/ckpt_epoch190.pt"
# checkpoint_path = "/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_DVAE_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_latent_2_date_191025_1906_trial_1/ckpt_epoch300.pt"
checkpoint_path = experiment_config.checkpoint_path

checkpoint = torch.load(checkpoint_path, map_location=torch.device(experiment_config.device), weights_only=False)
# model = TransformerDVAE(
#     experiment_config.model_config
# ).to(experiment_config.device)
model = TransformerDVAE(
    checkpoint["config"].model_config
).to(experiment_config.device)

model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
# test model reconstruction
x = val_trajectories[:500]  # [T] numpy array of token
x = torch.tensor(x, dtype=torch.long).to(experiment_config.device)  # [1, T]
C = model.context_window
x_unfolded = x.unfold(dimension=-1, size=C, step=1)
B, T, C_out = x_unfolded.shape
x_reshaped = x_unfolded.reshape(B * T, C_out)
model.eval()
with torch.no_grad():
    z, mu_q, logvar_q = model.inference(x_reshaped) # z, mu, logvar: [1*T, latent_dim]
    logits = model.decode(z) # logits: [1*T, vocab_size]
    reconstructed_ids = logits.argmax(dim=-1).cpu().numpy()  # [1*T]
    reconstructed_tokens = [NT_id2token[id] for id in reconstructed_ids]
    print("Original:     ", [NT_id2token[id] for id in x[0, C-1:].cpu().numpy()])
    print("Reconstructed:", reconstructed_tokens)

In [ ]:
x_reshaped.shape

In [ ]:
mask = x_reshaped[:, -1] == 2
# mask = torch.ones(x_reshaped.shape[0], device=experiment_config.device, dtype=torch.bool) # all tokens
memories = infer_memory_from_sequence(x_reshaped[:], M_ids=[2, 3])
M1_mask = (memories == 1)
M2_mask = (memories == 2)
combined_mask_1 = (mask * M1_mask).detach().cpu().numpy()
combined_mask_2 = (mask * M2_mask).detach().cpu().numpy()

In [ ]:
mu_q_numpy = mu_q.detach().cpu().numpy()

plt.figure()
plt.scatter(mu_q_numpy[combined_mask_1, 0], mu_q_numpy[combined_mask_1, 1], label="Mu (Memory 1)")
plt.scatter(mu_q_numpy[combined_mask_2, 0], mu_q_numpy[combined_mask_2, 1], label="Mu (Memory 2)")
plt.grid()
plt.xlabel("Latent Dim 1")
plt.ylabel("Latent Dim 2")
plt.legend()

fig = plt.figure(figsize=(15, 12))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(
    mu_q_numpy[combined_mask_1, 0],
    mu_q_numpy[combined_mask_1, 1],
    mu_q_numpy[combined_mask_1, 2],
    label="Mask 1"
)
ax.scatter(
    mu_q_numpy[combined_mask_2, 0],
    mu_q_numpy[combined_mask_2, 1],
    mu_q_numpy[combined_mask_2, 2],
    label="Mask 2"
)

ax.set_xlabel("Latent Dim 1")
ax.set_ylabel("Latent Dim 2")
ax.set_zlabel("Latent Dim 3")
ax.legend()

plt.show()

In [ ]:
(mask * M2_mask).sum()

In [ ]:
model.eval()
with torch.no_grad():
    enc_out = model.encoder(x_reshaped, n_layers=4, use_head=False) # (B, T, E)
# last_token_activations = enc_out[:, -1] # [B, E]
# last_token_activations = enc_out[mask * M1_mask, -1:].reshape(x_reshaped[mask * M1_mask].shape[0], -1) # [B, E]
# last_token_activations2 = enc_out[mask * M2_mask, -1:].reshape(x_reshaped[mask * M2_mask].shape[0], -1) # [B, E]
last_token_activations = enc_out[:, -1:].reshape(x_reshaped[:].shape[0], -1) # [B, E]

In [ ]:
probs = F.softmax(enc_out[:,-1], dim=-1).cpu().detach().numpy()
plt.figure()
plt.plot(probs[:50], 'o')

In [ ]:
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D projection

pca = PCA(n_components=32)

embeddings = pca.fit_transform(last_token_activations.cpu().detach().numpy())
# embeddings = pca.transform(last_token_activations.cpu().detach().numpy())
# embeddings2 = pca.transform(last_token_activations2.cpu().detach().numpy())
# mu_p_pca = pca.transform(last_token_activations.cpu().numpy())

plt.figure(figsize=(8, 5))
plt.scatter(embeddings[combined_mask_1, 0], embeddings[combined_mask_1, 1], marker='o', label='Mask 1')
plt.scatter(embeddings[combined_mask_2, 0], embeddings[combined_mask_2, 1], marker='o', label='Mask 2')
# plt.scatter(embeddings[M1_mask, 0], embeddings[M1_mask, 1], marker='o', label='Mask 1')
# plt.scatter(embeddings[M2_mask, 0], embeddings[M2_mask, 1], marker='o', label='Mask 2')
# plt.scatter(embeddings2[:, 0], embeddings2[:, 1], marker='o', label='Encoder Mu (GT)')
# plt.scatter(mu_p_pca[:, 0], mu_p_pca[:, 1], marker='o', label='Transition Mu')
# plt.plot([0, 2], [0, 2], linestyle='-', label="x=y")
# plt.plot(steps, mu_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='Mu Transition')
# plt.plot(steps, std_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='STD Transition')
# plt.title(f'PCA of Mu following {NT_id2token[previous_token_id]}, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
plt.title(f'PCA of Last Layer Embeddings, Explained Variance: {pca.explained_variance_ratio_[:2].sum():.3f}')
plt.xlabel('PC1')
plt.ylabel('PC2')
# plt.xticks(steps)
plt.grid(True)
plt.legend()
plt.axis("equal")

fig = plt.figure(figsize=(15, 12))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(
    embeddings[combined_mask_1, 0],
    embeddings[combined_mask_1, 1],
    embeddings[combined_mask_1, 2],
    label="Mask 1"
)
ax.scatter(
    embeddings[combined_mask_2, 0],
    embeddings[combined_mask_2, 1],
    embeddings[combined_mask_2, 2],
    label="Mask 2"
)
# ax.scatter(
#     embeddings[M1_mask, 0],
#     embeddings[M1_mask, 1],
#     embeddings[M1_mask, 2],
#     label="Mask 1"
# )
# ax.scatter(
#     embeddings[M2_mask, 0],
#     embeddings[M2_mask, 1],
#     embeddings[M2_mask, 2],
#     label="Mask 2"
# )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title(f'PCA of Mu, Explained Variance: {pca.explained_variance_ratio_[:3].sum():.3f}')
ax.legend()

plt.show()

In [ ]:
offset = 6
# z0 = z[offset:offset+1]
z0 = mu_q[offset:offset+1]

In [ ]:
latent_trajectory, decoded_logits = model.generate_decode_latent_trajectory(seq_len=100, z0=z0, do_reparameterization=True, device=experiment_config.device)

In [ ]:
# decode logits to tokens
decoded_ids = decoded_logits.argmax(dim=-1).cpu().numpy()  # [B, seq_len]

decoded_tokens = [NT_id2token[id] for id in decoded_ids[0]]

In [ ]:
print(decoded_tokens)

In [ ]:
def calculate_multistep_prediction_accuracy(model, dataloader, max_steps, device, pad_id):
    """
    Calculates the accuracy for 0-step, 1-step, ..., max_steps-step predictions
    by averaging over all possible starting windows in the dataset.
    """
    model.eval()
    
    # Store total correct predictions for each step
    correct_predictions = [0] * (max_steps + 1)
    # Store count of valid (non-padded) tokens for each step
    token_counts = [0] * (max_steps + 1)

    with torch.no_grad():
        for x in tqdm(dataloader, desc="Evaluating Multi-step Accuracy"):
            x = x.to(device)
            B, T_data = x.shape
            C = model.context_window

            # We need at least C + max_steps tokens to make a max_steps prediction
            if T_data < C + max_steps:
                continue
            
            # Unfold to get sliding windows. Step is 1.
            x_unfolded = x.unfold(dimension=-1, size=C, step=1)
            num_windows = x_unfolded.size(1)

            # Iterate over all possible starting windows `t`
            for t in range(num_windows - max_steps):
                # Get the initial latent state by encoding the window at time `t`
                initial_window = x_unfolded[:, t, :]
                z_current, _, _ = model.inference(initial_window)

                # Iterate through the prediction steps (0-step, 1-step, etc.)
                for k in range(max_steps + 1):
                    # For k > 0, advance the latent state using the transition model
                    if k > 0:
                        mu_p, logvar_p = model.transition_model(z_current)
                        L_t = model.build_cholesky_L(logvar_p)
                        dist_t = MultivariateNormal(loc=mu_p, scale_tril=L_t)
                        z_current = dist_t.rsample()

                    # Decode the (potentially predicted) latent state to get logits
                    logits = model.decode(z_current) # Shape: [B, vocab_size]
                    # logits = model.decode(mu_p) if k > 0 else model.decode(z_current) # Shape: [B, vocab_size]
                    
                    # Get the predicted token by taking the argmax of the logits
                    predicted_tokens = torch.argmax(logits, dim=1) # Shape: [B]
                    
                    # The target for a k-step prediction starting from window `t`
                    # is the last token of window `t+k`.
                    target_window = x_unfolded[:, t + k, :]
                    targets = target_window[:, -1] # Shape: [B]

                    # Mask out padding tokens
                    mask = (targets != pad_id)
                    
                    # Accumulate correct predictions and count for valid tokens
                    correct_predictions[k] += (predicted_tokens[mask] == targets[mask]).sum().item()
                    token_counts[k] += mask.sum().item()

    # Calculate average accuracy for each step
    avg_accuracies = [correct_predictions[k] / token_counts[k] if token_counts[k] > 0 else 0.0 
                      for k in range(max_steps + 1)]
    
    return avg_accuracies

def calculate_multistep_prediction_loss(model, dataloader, max_steps, device, pad_id):
    """
    Calculates the cross-entropy loss for 0-step, 1-step, ..., max_steps-step predictions
    by averaging over all possible starting windows in the dataset.
    """
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id, reduction='none')
    
    # Store total loss for each step
    total_losses = [0.0] * (max_steps + 1)
    # Store count of valid (non-padded) tokens for each step
    token_counts = [0] * (max_steps + 1)

    with torch.no_grad():
        for x in tqdm(dataloader, desc="Evaluating Multi-step Loss"):
            x = x.to(device)
            B, T_data = x.shape
            C = model.context_window

            # We need at least C + max_steps tokens to make a max_steps prediction
            if T_data < C + max_steps:
                continue
            
            # Unfold to get sliding windows. Step is 1.
            # x_unfolded has shape [B, T, C] where T = T_data - C + 1
            x_unfolded = x.unfold(dimension=-1, size=C, step=1)
            num_windows = x_unfolded.size(1)

            # Iterate over all possible starting windows `t`
            # The last possible start window is `num_windows - 1 - max_steps`
            for t in range(num_windows - max_steps):
                # Get the initial latent state by encoding the window at time `t`
                initial_window = x_unfolded[:, t, :]
                z_current, _, _ = model.inference(initial_window)

                # Iterate through the prediction steps (0-step, 1-step, etc.)
                for k in range(max_steps + 1):
                    # For k > 0, advance the latent state using the transition model
                    if k > 0:
                        mu_p, logvar_p = model.transition_model(z_current)
                        # std = torch.exp(0.5 * logvar_p)
                        # eps = torch.randn_like(std)
                        # z_current = mu_p + eps * std
                        L_t = model.build_cholesky_L(logvar_p)
                        dist_t = MultivariateNormal(loc=mu_p, scale_tril=L_t)
                        z_current = dist_t.rsample()

                    # Decode the (potentially predicted) latent state to get logits
                    logits = model.decode(z_current) # Shape: [B, vocab_size]
                    
                    # The target for a k-step prediction starting from window `t`
                    # is the last token of window `t+k`.
                    target_window = x_unfolded[:, t + k, :]
                    targets = target_window[:, -1] # Shape: [B]

                    # Calculate loss for the current step k
                    loss = criterion(logits, targets)
                    
                    # Mask out padding tokens
                    mask = (targets != pad_id)
                    
                    # Accumulate loss and count for valid tokens
                    total_losses[k] += loss[mask].sum().item()
                    token_counts[k] += mask.sum().item()

    # Calculate average cross-entropy for each step
    avg_losses = [total_losses[k] / token_counts[k] if token_counts[k] > 0 else 0.0 
                  for k in range(max_steps + 1)]
    
    return avg_losses

# --- Execution ---
# We can use the validation loader
prediction_loader = val_loader 
max_prediction_steps = 10

# Calculate the losses
multistep_losses = calculate_multistep_prediction_loss(
    model=model,
    dataloader=prediction_loader,
    max_steps=max_prediction_steps,
    device=experiment_config.device,
    pad_id=pad_id
)

multistep_accuracies = calculate_multistep_prediction_accuracy(
    model=model,
    dataloader=prediction_loader,
    max_steps=max_prediction_steps,
    device=experiment_config.device,
    pad_id=pad_id
)

# --- Plotting Results ---
import matplotlib.pyplot as plt

steps = range(max_prediction_steps + 1)
plt.figure(figsize=(8, 5))
plt.plot(steps, multistep_losses, marker='o', linestyle='-')
plt.title('Multi-step Prediction Cross-Entropy Loss')
plt.xlabel('Number of Transition Steps (k)')
plt.ylabel('Average Cross-Entropy Loss')
plt.xticks(steps)
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(steps, multistep_accuracies, marker='o', linestyle='-')
plt.title('Multi-step Prediction Accuracy')
plt.xlabel('Number of Transition Steps (k)')
plt.ylabel('Average Accuracy')
plt.xticks(steps)
plt.grid(True)
plt.show()

for i, loss in enumerate(multistep_losses):
    print(f"Step {i} Prediction Loss: {loss:.4f}")

for i, accuracy in enumerate(multistep_accuracies):
    print(f"Step {i} Prediction Accuracy: {accuracy:.4f}")

In [ ]:
def visualize_mu(model, dataloader, max_steps, device, previous_token_id, diagonal_variance=False):
    """
    
    """
    model.eval()
    
    mu_q_list = []
    mu_p_list = []
    std_p_list = []
    std_q_list = []

    with torch.no_grad():
        for x in tqdm(dataloader, desc="COllecting Mu and STD data"):
            x = x.to(device)
            B, T_data = x.shape
            C = model.context_window

            # We need at least C + max_steps tokens to make a max_steps prediction
            if T_data < C + max_steps:
                continue
            
            # Unfold to get sliding windows. Step is 1.
            # x_unfolded has shape [B, T, C] where T = T_data - C + 1
            x_unfolded = x.unfold(dimension=-1, size=C, step=1)
            num_windows = x_unfolded.size(1)

            # Iterate over all possible starting windows `t`
            # The last possible start window is `num_windows - 1 - max_steps`
            for t in range(1, num_windows - max_steps):
                # Get the initial latent state by encoding the window at time `t`
                initial_window = x_unfolded[:, t, :]
                z_current, mu_q, logvar_q = model.inference(initial_window)
                if x_unfolded[0, t-1, -1] in previous_token_id and t > 1:
                # if x_unfolded[0, t, -1] == previous_token_id and t > 1:
                    mu_q_list.append(mu_q)
                    if not diagonal_variance:
                        L_q = model.build_cholesky_L(logvar_q)
                        std_q_list.append(torch.stack([torch.diag(L_b) for L_b in L_q]))
                    else:
                        std_q_list.append(torch.sqrt(torch.exp(logvar_q)))

                # Iterate through the prediction steps (0-step, 1-step, etc.)
                for k in range(1, max_steps + 1):
                    # For k > 0, advance the latent state using the transition model
                    if k > 0:
                        mu_p, logvar_p = model.transition_model(z_current)
                        # std = torch.exp(0.5 * logvar_p)
                        # eps = torch.randn_like(std)
                        # z_current = mu_p + eps * std
                        L_t = model.build_cholesky_L(logvar_p)
                        dist_t = MultivariateNormal(loc=mu_p, scale_tril=L_t)
                        z_current = dist_t.rsample()

                    if k > 0:
                        if x_unfolded[0, t, -1] in previous_token_id and t < num_windows - max_steps - 1:
                            mu_p_list.append(mu_p)
                            std_p_list.append(torch.stack([torch.diag(L_b) for L_b in L_t]))

        
    
    return mu_q_list, mu_p_list, std_p_list, std_q_list

viz_loader = DataLoader(
    val_trajectory_dataset[:100], 
    # batch_size=experiment_config.training_config.batch_size, 
    batch_size=1,
    shuffle=False, 
    collate_fn=collate_fn_np, 
    drop_last=True,
    # num_workers=16,
    # prefetch_factor=2,
    # persistent_workers=True,
    # pin_memory=True
    )

# previous_token_id = [NT_token2id['D1']]
previous_token_id = [1]
mu_q_list, mu_p_list, std_p_list, std_q_list = visualize_mu(
    model=model,
    dataloader=viz_loader,
    max_steps=1,
    device=experiment_config.device,
    previous_token_id=previous_token_id,
    diagonal_variance=True
)

In [ ]:
mu_q = torch.stack(mu_q_list[:], dim=1)
std_q = torch.stack(std_q_list[:], dim=1)
mu_p = torch.stack(mu_p_list[:], dim=1)
std_p = torch.stack(std_p_list[:], dim=1)

In [ ]:
print(mu_q.shape)
print(mu_p.shape)

In [ ]:
n_examples = 200
steps = range(mu_q[:,:n_examples,:].shape[1])
for dim in range(model.latent_dim):
    plt.figure(figsize=(8, 5))
    plt.plot(steps, mu_q[0, :n_examples, dim].cpu().numpy(), marker='o', linestyle='-', label='Mu Encoder')
    plt.plot(steps, mu_p[0, :n_examples, dim].cpu().numpy(), marker='o', linestyle='-', label='Mu Transition')
    plt.plot(steps, std_q[0, :n_examples, dim].cpu().numpy(), marker='o', linestyle='-', label='STD Encoder')
    plt.plot(steps, std_p[0, :n_examples, dim].cpu().numpy(), marker='o', linestyle='-', label='STD Transition')
    # plt.title(f'1-Step Prediction Mu and STD, following {NT_id2token[previous_token_id]}, Latent dim {dim}')
    plt.title(f'1-Step Prediction Mu and STD, Latent dim {dim}')
    plt.xlabel('Token Index')
    # plt.ylabel('Average Cross-Entropy Loss')
    plt.xticks(steps)
    plt.grid(True)
    plt.legend()
    
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(mu_q[0, :, 0].cpu().numpy(), mu_q[0, :, 1].cpu().numpy(), marker='o', label='Encoder Mu (GT)')
plt.scatter(mu_p[0, :, 0].cpu().numpy(), mu_p[0, :, 1].cpu().numpy(), marker='o', label='Transition Mu')
# plt.plot([0, 2], [0, 2], linestyle='-', label="x=y")
# plt.plot(steps, mu_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='Mu Transition')
# plt.plot(steps, std_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='STD Transition')
# plt.title(f'PCA of Mu following {NT_id2token[previous_token_id]}, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
# plt.title(f'PCA of Mu, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
plt.xlabel('Latent Dim 1')
plt.ylabel('Latent Dim 2')
# plt.xticks(steps)
plt.grid(True)
plt.legend()
# plt.axis("equal")

In [ ]:
# plt.figure(figsize=(8, 5))
# plt.scatter(mu_q[0, :20, 0].cpu().numpy(), mu_p[0, :20, 0].cpu().numpy(), marker='o', label='Mu Encoder')
# plt.plot([0, 2], [0, 2], linestyle='-', label="x=y")
# # plt.plot(steps, mu_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='Mu Transition')
# # plt.plot(steps, std_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='STD Transition')
# plt.title(f'1-Step Prediction Mu and STD, Latent dim {0}')
# plt.xlabel('Encoder Mu (GT)')
# plt.ylabel('Transition Mu')
# # plt.xticks(steps)
# plt.grid(True)
# plt.legend()
# plt.axis("equal")
# plt.show()

In [ ]:
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D projection

pca = PCA(n_components=2)

mu_q_pca = pca.fit_transform(mu_q[0].cpu().detach().numpy())
mu_p_pca = pca.transform(mu_p[0].cpu().numpy())

plt.figure(figsize=(8, 5))
plt.scatter(mu_q_pca[:, 0], mu_q_pca[:, 1], marker='o', label='Encoder Mu (GT)')
plt.scatter(mu_p_pca[:, 0], mu_p_pca[:, 1], marker='o', label='Transition Mu')
# plt.plot([0, 2], [0, 2], linestyle='-', label="x=y")
# plt.plot(steps, mu_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='Mu Transition')
# plt.plot(steps, std_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='STD Transition')
# plt.title(f'PCA of Mu following {NT_id2token[previous_token_id]}, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
plt.title(f'PCA of Mu, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
plt.xlabel('PC1')
plt.ylabel('PC2')
# plt.xticks(steps)
plt.grid(True)
plt.legend()
plt.axis("equal")

if pca.n_components_ > 2:
    fig = plt.figure(figsize=(15, 12))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(
        mu_q_pca[:, 0],
        mu_q_pca[:, 1],
        mu_q_pca[:, 2],
        label="Encoder Mu (GT)"
    )
    ax.scatter(
        mu_p_pca[:, 0],
        mu_p_pca[:, 1],
        mu_p_pca[:, 2],
        label='Transition Mu'
    )
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_zlabel("PC3")
    ax.set_title(f'PCA of Mu, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
    ax.legend()

plt.show()

In [ ]:
print(pca.explained_variance_ratio_)
print(pca.explained_variance_ratio_.sum())

In [ ]:
bla = torch.Tensor([[0] * 32, [1] * 32, [2] * 32, [3] * 32, [4] * 32, [5] * 32, [6] * 32, [7] * 32])
z_current, mu_q, logvar_q = model.inference(bla.to(mu_q.device).long())

In [ ]:


from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D projection

pca = PCA(n_components=2)

# latent_trajectory_pca = pca.fit_transform(latent_trajectory[0].cpu().detach().numpy())
latent_trajectory_pca = latent_trajectory[0].cpu().detach().numpy()
# mu_p_pca = pca.transform(mu_p[0].cpu().numpy())
offset=3

plt.figure(figsize=(8, 5))
plt.scatter(latent_trajectory_pca[:, 0], latent_trajectory_pca[:, 1], marker='o', label='Latent Trajectory')
# plt.scatter(latent_trajectory[0, offset:offset+2, 0].cpu().detach().numpy(), latent_trajectory[0, offset:offset+2, 1].cpu().detach().numpy(), marker='o', label='Latent Trajectory')
# plt.scatter(mu_p_pca[:, 0], mu_p_pca[:, 1], marker='o', label='Transition Mu')
# plt.plot([0, 2], [0, 2], linestyle='-', label="x=y")
# plt.plot(steps, mu_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='Mu Transition')
# plt.plot(steps, std_p[0, :20, dim].cpu().numpy(), marker='o', linestyle='-', label='STD Transition')
# plt.title(f'PCA of Mu following {NT_id2token[previous_token_id]}, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
# plt.title(f'PCA of Mu, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
plt.xlabel('PC1')
plt.ylabel('PC2')
# plt.xticks(steps)
plt.grid(True)
plt.legend()
# plt.axis("equal")
# plt.xlim((-3, 3))
# plt.ylim((-3, 3))

# fig = plt.figure(figsize=(15, 12))
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(
#     mu_q_pca[:, 0],
#     mu_q_pca[:, 1],
#     mu_q_pca[:, 2],
#     label="Encoder Mu (GT)"
# )
# ax.scatter(
#     mu_p_pca[:, 0],
#     mu_p_pca[:, 1],
#     mu_p_pca[:, 2],
#     label='Transition Mu'
# )
# ax.set_xlabel("PC1")
# ax.set_ylabel("PC2")
# ax.set_zlabel("PC3")
# ax.set_title(f'PCA of Mu, Explained Variance: {pca.explained_variance_ratio_.sum():.3f}')
# ax.legend()

plt.show()

In [ ]:
steps = range(latent_trajectory[:,:40,:].shape[1])
for dim in range(model.latent_dim):
    plt.figure(figsize=(8, 5))
    plt.plot(steps, latent_trajectory[0, :40, dim].cpu().numpy(), marker='o', linestyle='-', label='Latent Trajectory')
    # plt.title(f'1-Step Prediction Mu and STD, following {NT_id2token[previous_token_id]}, Latent dim {dim}')
    # plt.title(f'1-Step Prediction Mu and STD, Latent dim {dim}')
    plt.xlabel('Token Index')
    # plt.ylabel('Average Cross-Entropy Loss')
    plt.xticks(steps)
    plt.grid(True)
    plt.legend()
    
plt.show()

In [ ]:
print(decoded_tokens)

In [ ]:
### Dynamical VAE Fixed Encoder Training
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        # Training metrics
        self.train_total_losses = []
        self.train_reconstruction_losses = []
        self.train_kl_losses = []
        self.train_accuracies = []
        
        # Validation metrics
        self.val_reconstruction_losses = []
        self.val_kl_losses = []
        self.val_accuracies = []
        
        self.epochs = []
        self.learning_rates = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_total_losses = metrics_dict.get('train_total_losses', [])
        self.train_reconstruction_losses = metrics_dict.get('train_reconstruction_losses', [])
        self.train_kl_losses = metrics_dict.get('train_kl_losses', [])
        self.train_accuracies = metrics_dict.get('train_accuracies', [])
        
        self.val_reconstruction_losses = metrics_dict.get('val_reconstruction_losses', [])
        self.val_kl_losses = metrics_dict.get('val_kl_losses', [])
        self.val_accuracies = metrics_dict.get('val_accuracies', [])
        self.learning_rates = metrics_dict.get('learning_rates', [])
        
        self.epochs = list(range(1, len(self.train_total_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Color scheme
        train_color = "#287BBB"  # Blue
        val_color = "#ED9E00"    # Orange
        
        # Plot 1: Reconstruction losses
        axes[0,0].plot(self.epochs, self.train_reconstruction_losses, 
                    color=train_color, linestyle='-', label='Train Reconstruction')
        axes[0,0].plot(self.epochs, self.val_reconstruction_losses, 
                    color=val_color, linestyle='-', label='Val Reconstruction')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Loss')
        axes[0,0].set_title('Reconstruction Loss')
        axes[0,0].legend()
        axes[0,0].grid(True)

        # Plot 2: KL losses
        axes[0,1].plot(self.epochs, self.train_kl_losses, 
                    color=train_color, linestyle='-', label='Train KL')
        axes[0,1].plot(self.epochs, self.val_kl_losses, 
                    color=val_color, linestyle='-', label='Val KL')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Loss')
        axes[0,1].set_title('KL Divergence Loss')
        axes[0,1].legend()
        axes[0,1].grid(True)

        # Plot 3: Total training loss and LR
        ax3_twin = axes[1,0].twinx()
        axes[1,0].plot(self.epochs, self.train_total_losses, color=train_color, label='Total Train Loss')
        ax3_twin.plot(self.epochs, self.learning_rates, color='green', linestyle='--', label='Learning Rate')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Loss')
        ax3_twin.set_ylabel('Learning Rate')
        axes[1,0].set_title('Total Training Loss and Learning Rate')
        axes[1,0].legend(loc='upper left')
        ax3_twin.legend(loc='upper right')
        axes[1,0].grid(True)


        # Plot 4: Accuracies
        axes[1,1].plot(self.epochs, self.train_accuracies, 
                    color=train_color, linestyle='-', label='Train Accuracy')
        axes[1,1].plot(self.epochs, self.val_accuracies, 
                    color=val_color, linestyle='-', label='Val Accuracy')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Accuracy')
        axes[1,1].set_title('Model Accuracy')
        axes[1,1].legend()
        axes[1,1].grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png", dpi=300, bbox_inches='tight')

        plt.show()

def load_checkpoint(checkpoint_path, model, optimizer, scheduler, experiment_config, device):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch, loaded metrics, and global step
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler state if provided
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    # Compare relevant parts of the model config
    current_model_config = experiment_config.model_config
    saved_model_config = saved_config.model_config
    
    mismatched_keys = []
    for key in ['n_layers', 'n_heads', 'embed_dim', 'ffn_dim', 'context_window', 'latent_dim']:
        if getattr(current_model_config, key) != getattr(saved_model_config, key):
            mismatched_keys.append(key)
    
    if mismatched_keys:
        raise ValueError(f"Checkpoint config mismatch for keys: {mismatched_keys}")

    starting_epoch = checkpoint['epoch']
    global_step = checkpoint.get('global_step', 0)
    return starting_epoch, checkpoint['metrics'], global_step

n_trials = 1
for trial in range(n_trials):
    # Initialize model
    if experiment_config.model_config.mode == 'DVAE':
        model = TransformerDVAEFixedEncoder(
            experiment_config.model_config
        ).to(experiment_config.device)

    else:
        raise ValueError(f"Unsupported mode: {experiment_config.model_config.mode}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=experiment_config.training_config.lr)
    scheduler = None
    if experiment_config.training_config.use_scheduler:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=experiment_config.training_config.scheduler_T_max,
            eta_min=experiment_config.training_config.scheduler_eta_min
        )
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, experiment_config.model_save_prefix + f"FIXED_ENCODER_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    global_step = 0
    if experiment_config.checkpoint_path:
        try:
            starting_epoch, saved_metrics, global_step = load_checkpoint(
                experiment_config.checkpoint_path,
                model,
                optimizer,
                scheduler,
                experiment_config,
                experiment_config.device
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = experiment_config.training_config.lr
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}, global_step {global_step}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0
            global_step = 0

    print("Starting training loop...")
    global_start = time.time()

    # Load a pretrained LM model to use its encoder
    # Loading a saved model
    lm_ckpt_path = experiment_config.LM_checkpoint_path
    # ckpt_path = "/content/tiny_transformer_dyck2_layers_2_embed_32_context_window_64_weight_tied_maxdepth_20_max_len_50_date_29_07_25_trial_1/model_epoch15.pt"
    # if lm_ckpt_path:
    #     LM_checkpoint = torch.load(lm_ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)

    #     if LM_checkpoint['config'].model_config.mode == 'LM':
    #         lm_model = TinyLlamaTransformer(
    #             LM_checkpoint['config'].model_config
    #         ).to(experiment_config.device)
    #     model.encoder = lm_model

    # for param in model.encoder.parameters():
    #     param.requires_grad = False

    def validate(model: TransformerDVAE, val_loader, criterion, device):
        model.eval()
        total_reconstruction_loss = 0
        total_kl_loss = 0
        total_acc = 0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device)
                
                # x is [B, T] tensor of token trajectories. Virtually stack it into [B, T', C] windows
                B, T_data = x.shape
                C = model.context_window
                x_unfolded = x.unfold(dimension=-1, size=C, step=1)
                B, T, C_out = x_unfolded.shape
                
                x_reshaped = x_unfolded.reshape(B * T, C_out)
                
                # inference: encode observations into latents
                z, mu_q, logvar_q = model.inference(x_reshaped) # z, mu, logvar: [B*T, latent_dim]

                # decode the latents to reconstruct the observations
                logits = model.decode(z) # logits: [B*T, vocab_size]

                loss_reconstruction = criterion(logits, x_reshaped[:, -1])
                acc = calculate_accuracy(logits, x_reshaped[:, -1], pad_id)

                # reshape z to [B, T, latent_dim] for KL calculation
                z = z.reshape(B, T, model.latent_dim)
                
                # initial_latent = torch.zeros((B, model.latent_dim), device=device)
                # mu_p_t0, logvar_p_t0 = model.transition_model(initial_latent, is_t0=True)  # [B, latent_dim]
                # z_prev = torch.cat([initial_latent.unsqueeze(1), z[:, :-1, :]], dim=1)  # [B, T, latent_dim]
                z_prev = z[:, :-1, :]  # [B, T-1, latent_dim]
                mu_p, logvar_p = model.transition_model(z_prev.reshape(B*(T-1), -1))  # [B*(T-1), latent_dim]

                # # concatenate t=0 prior
                # mu_p = mu_p.reshape(B, T-1, -1)
                # logvar_p = logvar_p.reshape(B, T-1, -1)
                # mu_p = torch.cat([mu_p_t0.unsqueeze(1), mu_p], dim=1).reshape(B*T, -1)
                # logvar_p = torch.cat([logvar_p_t0.unsqueeze(1), logvar_p], dim=1).reshape(B*T, -1)

                # remove first posterior from each trajectory (it cannot be predicted) to match prior shape
                mu_q = mu_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)
                logvar_q = logvar_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)

                # # construct L matrix for transition priors
                # var_p_log_diag = logvar_p[:, :model.latent_dim]
                # var_p_off_diag = logvar_p[:, model.latent_dim:]
                # positive_diag = torch.exp(var_p_log_diag)
                # L = torch.zeros(logvar_p.shape[0], model.latent_dim, model.latent_dim, device=logvar_q.device)
                # tril_indices = torch.tril_indices(row=model.latent_dim, col=model.latent_dim, offset=-1)
                # L[:, tril_indices[0], tril_indices[1]] = var_p_off_diag
                # L += torch.diag_embed(positive_diag)

                # L_q = model.build_cholesky_L(logvar_q)
                L_p = model.build_cholesky_L(logvar_p)

                dist_q = MultivariateNormal(loc=mu_q, covariance_matrix=torch.diag_embed(torch.exp(logvar_q)))
                # dist_q = MultivariateNormal(loc=mu_q, scale_tril=L_q)
                # dist_p = MultivariateNormal(loc=mu_p, covariance_matrix=torch.diag_embed(torch.exp(logvar_p)))
                dist_p = MultivariateNormal(loc=mu_p, scale_tril=L_p)

                KL_dist = kl_divergence(dist_q, dist_p)

                # Compute KL divergence loss
                # Dkl = kl_divergence_gaussians(mu_q, logvar_q, mu_p, logvar_p)  # [B*(T-1)]
                loss_kl = KL_dist.sum() / (B * (T-1) * model.latent_dim)

                total_reconstruction_loss += loss_reconstruction.item()
                total_kl_loss += loss_kl.item()
                total_acc += acc

                # predictions distributional statistics
                # mean_mu_q = mu_q.mean(dim=0).detach().cpu()
                # std_mu_q = mu_q.std(dim=0).detach().cpu()
                # mean_logvar_q = logvar_q.mean(dim=0).detach().cpu()
                # std_logvar_q = logvar_q.std(dim=0).detach().cpu()
                # mean_mu_p = mu_p.mean(dim=0).detach().cpu()
                # std_mu_p = mu_p.std(dim=0).detach().cpu()
                # mean_logvar_p = logvar_p.mean(dim=0).detach().cpu()
                # std_logvar_p = logvar_p.std(dim=0).detach().cpu()

                # correlation_matrix = torch.corrcoef(torch.stack((mean_mu_q, mean_mu_p)))

                # The Pearson correlation coefficient between x and y is at index [0, 1] or [1, 0]
                # pearson_r = correlation_matrix[0, 1]

                # print(f"val batch:")
                # print(f"mean_mu_q={mean_mu_q}, \nstd_mu_q={std_mu_q}, \nmean_logvar_q={mean_logvar_q}, \nstd_logvar_q={std_logvar_q}")
                # print(f"mean_mu_p={mean_mu_p}, \nstd_mu_p={std_mu_p}, \nmean_logvar_p={mean_logvar_p}, \nstd_logvar_p={std_logvar_p}")
                # print(f"mean_mu correlation: {pearson_r}")

        return total_reconstruction_loss / len(val_loader), total_kl_loss / len(val_loader), total_acc / len(val_loader)

    # Training loop
    device = experiment_config.device
    for epoch in range(starting_epoch, experiment_config.epochs):
        model.train()
        model.encoder.eval()  # keep LM encoder in eval mode
        epoch_loss = 0.0
        epoch_loss_reconstruction = 0.0
        epoch_loss_kl = 0.0
        epoch_acc = 0.0
        start = time.time()

        # Training phase
        for batch, x in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            x = x.to(device)
            
            # x is [B, T] tensor of token trajectories. Virtually stack it into [B, T', C] windows
            B, T_data = x.shape
            C = model.context_window
            x_unfolded = x.unfold(dimension=-1, size=C, step=1)
            B, T, C_out = x_unfolded.shape
            
            x_reshaped = x_unfolded.reshape(B * T, C_out)
            
            # inference: encode observations into latents
            z, mu_q, logvar_q = model.inference(x_reshaped) # z, mu, logvar: [B*T, latent_dim]

            # decode the latents to reconstruct the observations
            logits = model.decode(z) # logits: [B*T, vocab_size]

            loss_reconstruction = criterion(logits, x_reshaped[:, -1]) # reconstruction loss only for the last token in each window
            acc = calculate_accuracy(logits, x_reshaped[:, -1], pad_id)  # autoencoding reconstruction accuracy

            # reshape z to [B, T, latent_dim] for KL calculation
            z = z.reshape(B, T, model.latent_dim)

            # # TODO: add option to sample from initial prior instead of zero
            # initial_latent = torch.zeros((B, model.latent_dim), device=device)
            # mu_p_t0, logvar_p_t0 = model.transition_model(initial_latent, is_t0=True)  # [B, latent_dim]
            # z_prev = torch.cat([initial_latent.unsqueeze(1), z[:, :-1, :]], dim=1)  # [B, T, latent_dim]
            z_prev = z[:, :-1, :]  # [B, T-1, latent_dim]
            mu_p, logvar_p = model.transition_model(z_prev.reshape(B*(T-1), -1))  # [B*(T-1), latent_dim]

            # # concatenate t=0 prior
            # mu_p = mu_p.reshape(B, T-1, -1)
            # logvar_p = logvar_p.reshape(B, T-1, -1)
            # mu_p = torch.cat([mu_p_t0.unsqueeze(1), mu_p], dim=1).reshape(B*T, -1)
            # logvar_p = torch.cat([logvar_p_t0.unsqueeze(1), logvar_p], dim=1).reshape(B*T, -1)

            # remove first posterior from each trajectory (it cannot be predicted) to match prior shape
            mu_q = mu_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)
            logvar_q = logvar_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)

            # # construct L matrix for transition priors
            # var_p_log_diag = logvar_p[:, :model.latent_dim]
            # var_p_off_diag = logvar_p[:, model.latent_dim:]
            # positive_diag = torch.exp(var_p_log_diag)
            # L = torch.zeros(logvar_p.shape[0], model.latent_dim, model.latent_dim, device=logvar_q.device)
            # tril_indices = torch.tril_indices(row=model.latent_dim, col=model.latent_dim, offset=-1)
            # L[:, tril_indices[0], tril_indices[1]] = var_p_off_diag
            # L += torch.diag_embed(positive_diag)

            # L_q = model.build_cholesky_L(logvar_q)
            L_p = model.build_cholesky_L(logvar_p)

            dist_q = MultivariateNormal(loc=mu_q, covariance_matrix=torch.diag_embed(torch.exp(logvar_q)))
            # dist_q = MultivariateNormal(loc=mu_q, scale_tril=L_q)
            # dist_p = MultivariateNormal(loc=mu_p, covariance_matrix=torch.diag_embed(torch.exp(logvar_p)))
            dist_p = MultivariateNormal(loc=mu_p, scale_tril=L_p)

            KL_dist = kl_divergence(dist_q, dist_p)

            # Compute KL divergence loss
            # Dkl = kl_divergence_gaussians(mu_q, logvar_q, mu_p, logvar_p)  # [B*(T-1)]
            loss_kl = KL_dist.sum() / (B * (T-1) * model.latent_dim)

            # print(f"torch.kl: {KL_dist}, my KL: {Dkl}, KL loss: {loss_kl}")

            # compute KL-annealing beta
            beta = (experiment_config.training_config.maximal_beta - experiment_config.training_config.minimal_beta) * min(1.0, global_step / experiment_config.training_config.warmup_steps) + experiment_config.training_config.minimal_beta
            loss = experiment_config.training_config.reconstruction_coef * loss_reconstruction + beta * loss_kl
            # loss = beta * loss_reconstruction + experiment_config.training_config.reconstruction_coef * loss_kl

            optimizer.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            if experiment_config.training_config.grad_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            global_step += 1

            epoch_loss += loss.item()
            epoch_loss_reconstruction += loss_reconstruction.item()
            epoch_loss_kl += loss_kl.item()
            epoch_acc += acc

        # Validation phase
        val_reconstruction_loss, val_kl_loss, val_acc = validate(model, val_loader, criterion, device)

        # Calculate average losses and accuracies
        train_loss = epoch_loss / len(train_loader)
        train_acc = epoch_acc / len(train_loader)
        train_loss_reconstruction = epoch_loss_reconstruction / len(train_loader)
        train_loss_kl = epoch_loss_kl / len(train_loader)

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_total_losses.append(train_loss)
        metrics.train_reconstruction_losses.append(train_loss_reconstruction)
        metrics.train_kl_losses.append(train_loss_kl)
        metrics.train_accuracies.append(train_acc)
        metrics.val_reconstruction_losses.append(val_reconstruction_loss)
        metrics.val_kl_losses.append(val_kl_loss)
        metrics.val_accuracies.append(val_acc)
        metrics.learning_rates.append(optimizer.param_groups[0]['lr'])

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Total Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Train Reconstruction Loss: {train_loss_reconstruction:.4f} | Train KL Loss: {train_loss_kl:.4f}")
        print(f"Val Reconstruction Loss: {val_reconstruction_loss:.4f} | Val KL Loss: {val_kl_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")
        
        # Step the scheduler
        if scheduler:
            scheduler.step()
            print(f"LR: {scheduler.get_last_lr()[0]:.6f}")

        # Save checkpoint
        if (epoch+1) % experiment_config.save_every == 0 or (epoch+1) == experiment_config.epochs:
            # Plot metrics
            metrics.plot_metrics(save_dir)

            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            checkpoint_data = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': experiment_config,
                'epoch': epoch+1,
                'global_step': global_step,
                'metrics': {
                    'train_total_losses': metrics.train_total_losses,
                    'train_reconstruction_losses': metrics.train_reconstruction_losses,
                    'train_kl_losses': metrics.train_kl_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'val_reconstruction_losses': metrics.val_reconstruction_losses,
                    'val_kl_losses': metrics.val_kl_losses,
                    'val_accuracies': metrics.val_accuracies,
                    'learning_rates': metrics.learning_rates
                }
            }
            if scheduler:
                checkpoint_data['scheduler_state_dict'] = scheduler.state_dict()
            
            torch.save(checkpoint_data, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")


    print("Training complete!")

In [ ]:
model.encoder.embed.weight.data

In [ ]:
plt.Figure()
plt.scatter(model.encoder.embed.weight.data[:, 0].detach().cpu().numpy(), model.encoder.embed.weight.data[:, 1].detach().cpu().numpy())

In [ ]:
def compute_gradient_penalty(critic, real_samples, fake_samples, device):
                """Calculates the gradient penalty loss for WGAN-GP"""
                # Random weight term for interpolation between real and fake samples
                alpha = torch.rand((real_samples.size(0), 1, 1), device=device)
                # Get random interpolation between real and fake samples
                interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
                
                critic_interpolates = critic(interpolates).squeeze(-1) # [B]
                
                # Get gradient w.r.t. interpolates
                gradients = torch.autograd.grad(
                    outputs=critic_interpolates,
                    inputs=interpolates,
                    grad_outputs=torch.ones_like(critic_interpolates),
                    create_graph=True,
                    retain_graph=True,
                    # only_inputs=True # used by GPT
                )[0]
                
                gradients = gradients.view(gradients.size(0), -1)
                print(gradients.norm(2, dim=1).detach().mean().cpu().item())
                gradient_penalty = (torch.relu((gradients.norm(2, dim=1) - 1)) ** 2).mean()
                return gradient_penalty

In [ ]:
# DVAE-GAN config
model_config = TinyDVAEConfig(
    n_layers=4,
    n_heads=4,
    embed_dim=32,
    ffn_dim=256,
    context_window=32,
    vocab=NT_vocab,
    dropout_self_attention=0.05,
    dropout_embed=0.03,
    dropout_residual=0.03,
    dropout_latent=0.00,
    latent_dim=2,
    decoder_ffn_dim=64,
    dropout_decoder=0.03,
    transition_ffn_dim=64,
    dropout_transition=0.03,
    pooling='last',
    decoder_ln=False,
    transition_ln=False
)

critic_config = TinyLMConfig(
    n_layers=4,
    n_heads=1,
    embed_dim=model_config.latent_dim,
    # ffn_dim=model_config.latent_dim * 4,
    ffn_dim=64,
    context_window=200,
    vocab=["score"],
    dropout_self_attention=0.00,
    dropout_embed=0.00,
    dropout_residual=0.00,
)

training_config = TrainingDAVBConfig(
    lr=1e-4,
    lr_critic=1e-4,
    batch_size=64,
    grad_clipping=True,
    reconstruction_coef=1.0,
    warmup_steps=1000,
    minimal_beta=0.00,
    maximal_beta=0.0,
    n_encoder_layers=4,
    teacher_forcing=True,
    use_scheduler=True,
    scheduler_T_max=500,
    scheduler_eta_min=1e-5,
    gp_factor=10
)

experiment_config = ExperimentAVBConfig(
    epochs=500,
    checkpoint_path=None,
    # checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_DVAE_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_latent_5_date_041025_1133_trial_1/ckpt_epoch500.pt",
    save_every=2,
    device_index=0,
    model_config=model_config,
    critic_config=critic_config,
    critic_steps=5,
    training_config=training_config,
    LM_checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/221025/tiny_LM_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_date_221025_1107_trial_1/ckpt_epoch5.pt" # new simple memory (no noise) LM after fixed loading
)

pad_id = 100

# TODO: move the model save prefix to the ExperimentConfig initialization
current_date_ddmmyy = datetime.now().strftime('%d%m%y_%H%M')
current_date_only_day = datetime.now().strftime('%d%m%y')
if model_config.mode == 'AE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
if model_config.mode == 'KAE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_n_diagonals_{model_config.n_diagonals}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
if model_config.mode == 'DVAE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_date_{current_date_ddmmyy}"
elif model_config.mode == 'LM' or model_config.mode == 'RLM':
    # TODO: add memory generation parameters to name
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_date_{current_date_ddmmyy}"

experiment_config.model_save_prefix = os.path.join(current_date_only_day, experiment_config.model_save_prefix)

print(f"Using device: {experiment_config.device}")

In [ ]:
### Dynamical VAE-GAN Training
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

T_generation = 5

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        # Training metrics
        self.train_total_losses = []
        self.train_reconstruction_losses = []
        self.train_avb_losses = []  # Replaced KL with AVB
        self.train_critic_losses = []  # Added critic losses
        self.train_gradient_penalties = []  # Added gradient penalties
        self.train_q_scores = []  # Added q_scores
        self.train_p_scores = []  # Added p_scores
        self.train_accuracies = []
        
        # Validation metrics
        self.val_total_losses = []
        self.val_reconstruction_losses = []
        self.val_avb_losses = []  # Replaced KL with AVB
        self.val_critic_losses = []  # Added critic losses
        self.val_q_scores = []  # Added q_scores
        self.val_p_scores = []  # Added p_scores
        self.val_accuracies = []
        
        self.epochs = []
        self.learning_rates = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_total_losses = metrics_dict.get('train_total_losses', [])
        self.train_reconstruction_losses = metrics_dict.get('train_reconstruction_losses', [])
        self.train_avb_losses = metrics_dict.get('train_avb_losses', [])
        self.train_critic_losses = metrics_dict.get('train_critic_losses', [])
        self.train_gradient_penalties = metrics_dict.get('train_gradient_penalties', [])
        self.train_q_scores = metrics_dict.get('train_q_scores', [])
        self.train_p_scores = metrics_dict.get('train_p_scores', [])
        self.train_accuracies = metrics_dict.get('train_accuracies', [])
        
        self.val_total_losses = metrics_dict.get('val_total_losses', [])
        self.val_reconstruction_losses = metrics_dict.get('val_reconstruction_losses', [])
        self.val_avb_losses = metrics_dict.get('val_avb_losses', [])
        self.val_critic_losses = metrics_dict.get('val_critic_losses', [])
        self.val_q_scores = metrics_dict.get('val_q_scores', [])
        self.val_p_scores = metrics_dict.get('val_p_scores', [])
        self.val_accuracies = metrics_dict.get('val_accuracies', [])
        self.learning_rates = metrics_dict.get('learning_rates', [])
        
        self.epochs = list(range(1, len(self.train_total_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, axes = plt.subplots(3, 3, figsize=(18, 15))
        
        # Color scheme
        train_color = "#287BBB"  # Blue
        val_color = "#ED9E00"    # Orange
        
        # Plot 1: Reconstruction losses
        axes[0,0].plot(self.epochs, self.train_reconstruction_losses, 
                    color=train_color, linestyle='-', label='Train')
        axes[0,0].plot(self.epochs, self.val_reconstruction_losses, 
                    color=val_color, linestyle='-', label='Val')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Loss')
        axes[0,0].set_title('Reconstruction Loss')
        axes[0,0].legend()
        axes[0,0].grid(True)

        # Plot 2: AVB losses
        axes[0,1].plot(self.epochs, self.train_avb_losses, 
                    color=train_color, linestyle='-', label='Train')
        axes[0,1].plot(self.epochs, self.val_avb_losses, 
                    color=val_color, linestyle='-', label='Val')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Loss')
        axes[0,1].set_title('AVB Loss')
        axes[0,1].legend()
        axes[0,1].grid(True)
        
        # Plot 3: Critic losses
        axes[0,2].plot(self.epochs, self.train_critic_losses, 
                    color=train_color, linestyle='-', label='Train')
        axes[0,2].plot(self.epochs, self.val_critic_losses, 
                    color=val_color, linestyle='-', label='Val')
        axes[0,2].set_xlabel('Epoch')
        axes[0,2].set_ylabel('Loss')
        axes[0,2].set_title('Critic Loss')
        axes[0,2].legend()
        axes[0,2].grid(True)

        # Plot 4: Total training loss
        axes[1,0].plot(self.epochs, self.train_total_losses, color=train_color, label='Train')
        axes[1,0].plot(self.epochs, self.val_total_losses, color=val_color, label='Val')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Loss')
        axes[1,0].set_title('Total Loss')
        axes[1,0].legend()
        axes[1,0].grid(True)

        # Plot 5: Accuracies
        axes[1,1].plot(self.epochs, self.train_accuracies, 
                    color=train_color, linestyle='-', label='Train')
        axes[1,1].plot(self.epochs, self.val_accuracies, 
                    color=val_color, linestyle='-', label='Val')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Accuracy')
        axes[1,1].set_title('Model Accuracy')
        axes[1,1].legend()
        axes[1,1].grid(True)
        
        # Plot 6: Learning rate
        axes[1,2].plot(self.epochs, self.learning_rates, color='green', label='LR')
        axes[1,2].set_xlabel('Epoch')
        axes[1,2].set_ylabel('Learning Rate')
        axes[1,2].set_title('Learning Rate Schedule')
        axes[1,2].legend()
        axes[1,2].grid(True)

        # Plot 7: Gradient Penalties
        axes[2,0].plot(self.epochs, self.train_gradient_penalties, 
                    color=train_color, linestyle='-', label='Train')
        axes[2,0].set_xlabel('Epoch')
        axes[2,0].set_ylabel('Gradient Penalty')
        axes[2,0].set_title('Gradient Penalty')
        axes[2,0].legend()
        axes[2,0].grid(True)

        # Plot 8: Critic Scores (Q)
        axes[2,1].plot(self.epochs, self.train_q_scores, 
                    color=train_color, linestyle='-', label='Train Q')
        axes[2,1].plot(self.epochs, self.val_q_scores, 
                    color=val_color, linestyle='-', label='Val Q')
        axes[2,1].set_xlabel('Epoch')
        axes[2,1].set_ylabel('Score')
        axes[2,1].set_title('Q Scores')
        axes[2,1].legend()
        axes[2,1].grid(True)

        # Plot 9: Critic Scores (P)
        axes[2,2].plot(self.epochs, self.train_p_scores, 
                    color=train_color, linestyle='-', label='Train P')
        axes[2,2].plot(self.epochs, self.val_p_scores, 
                    color=val_color, linestyle='-', label='Val P')
        axes[2,2].set_xlabel('Epoch')
        axes[2,2].set_ylabel('Score')
        axes[2,2].set_title('P Scores')
        axes[2,2].legend()
        axes[2,2].grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png", dpi=300, bbox_inches='tight')

        plt.show()

def load_checkpoint(checkpoint_path, model, critic, optimizer_model, optimizer_critic, scheduler, experiment_config, device):
    """Load model and optimizer state from a checkpoint"""
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model states
    model.load_state_dict(checkpoint['model_state_dict'])
    critic.load_state_dict(checkpoint['critic_state_dict'])

    # Load optimizer states
    if optimizer_model is not None:
        optimizer_model.load_state_dict(checkpoint['optimizer_model_state_dict'])
    if optimizer_critic is not None:
        optimizer_critic.load_state_dict(checkpoint['optimizer_critic_state_dict'])
    
    # Load scheduler state if provided
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    current_model_config = experiment_config.model_config
    saved_model_config = saved_config.model_config
    
    mismatched_keys = []
    for key in ['n_layers', 'n_heads', 'embed_dim', 'ffn_dim', 'context_window', 'latent_dim']:
        if getattr(current_model_config, key) != getattr(saved_model_config, key):
            mismatched_keys.append(key)
    
    if mismatched_keys:
        raise ValueError(f"Checkpoint config mismatch for keys: {mismatched_keys}")

    starting_epoch = checkpoint['epoch']
    global_step = checkpoint.get('global_step', 0)
    return starting_epoch, checkpoint['metrics'], global_step

def validate(model, critic, val_loader, criterion, device):
    """Validation function matching training phase structure"""
    model.eval()
    critic.eval()
    total_loss = 0
    total_reconstruction_loss = 0
    total_avb_loss = 0
    total_critic_loss = 0
    total_p_score = 0
    total_q_score = 0
    total_acc = 0
    n_batches = 0
    
    with torch.no_grad():
        for x in val_loader:
            x = x.to(device)
            B, T_data = x.shape
            C = model.context_window
            x_unfolded = x.unfold(dimension=-1, size=C, step=1)
            B, T, C_out = x_unfolded.shape
            x_reshaped = x_unfolded.reshape(B * T, C_out)
            
            # Get encoder outputs
            z, mu_q, logvar_q = model.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers)
            z_q_trajectories = z.reshape(B, T, model.latent_dim)

            # Reconstruction
            logits = model.decode(z)
            loss_reconstruction = criterion(logits, x_reshaped[:, -1])
            acc = calculate_accuracy(logits, x_reshaped[:, -1], pad_id)

            # Generate prior trajectory and get critic scores
            # z_p_trajectories = model.generate_latent_trajectory(seq_len=T, device=device, z0=z_q_trajectories[:, 0, :])
            z_p_trajectories = model.generate_latent_trajectory(seq_len=T_generation, device=device, z0=z_q_trajectories[:, 0, :])
            q_scores = critic(z_q_trajectories[:, :T_generation]).squeeze(-1)
            p_scores = critic(z_p_trajectories).squeeze(-1)

            # Calculate losses
            avb_loss = -p_scores.mean() - q_scores.mean()
            # gradient_penalty = compute_gradient_penalty(critic, z_q_trajectories, z_p_trajectories, device)
            # loss_critic = p_scores.mean() - q_scores.mean() + 10 * gradient_penalty
            loss_critic = p_scores.mean() - q_scores.mean()

            # Total loss (matching training phase computation)
            beta = (experiment_config.training_config.maximal_beta - experiment_config.training_config.minimal_beta) * \
                   min(1.0, global_step / experiment_config.training_config.warmup_steps) + \
                   experiment_config.training_config.minimal_beta
            loss = experiment_config.training_config.reconstruction_coef * loss_reconstruction + beta * avb_loss

            # Accumulate batch metrics
            total_loss += loss.item()
            total_reconstruction_loss += loss_reconstruction.item()
            total_avb_loss += avb_loss.item()
            total_critic_loss += loss_critic.item()
            total_p_score += p_scores.mean().item()
            total_q_score += q_scores.mean().item()
            total_acc += acc
            n_batches += 1

    return (total_loss / n_batches,
            total_reconstruction_loss / n_batches,
            total_avb_loss / n_batches,
            total_critic_loss / n_batches,
            total_p_score / n_batches,
            total_q_score / n_batches,
            total_acc / n_batches)

n_trials = 1
for trial in range(n_trials):
    # Initialize model
    if experiment_config.model_config.mode == 'DVAE':
        model = TransformerDVAE(
            experiment_config.model_config
        ).to(experiment_config.device)
    else:
        raise ValueError(f"Unsupported mode: {experiment_config.model_config.mode}")
    
    # Load pretrained LM model for encoder if specified
    lm_ckpt_path = experiment_config.LM_checkpoint_path
    if lm_ckpt_path:
        LM_checkpoint = torch.load(lm_ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)
        if LM_checkpoint['config'].model_config.mode == 'LM':
            lm_model = TinyLlamaTransformer(LM_checkpoint['config'].model_config).to(experiment_config.device)
        lm_model.load_state_dict(LM_checkpoint['model_state_dict'])
        model.encoder = lm_model
        for param in model.encoder.parameters():
            param.requires_grad = False
            
    # Initialize Critic
    critic = TinyLlamaCritic(experiment_config.critic_config).to(experiment_config.device)

    # Setup optimizers and scheduler
    model_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer_model = torch.optim.AdamW(model_params, lr=experiment_config.training_config.lr)
    optimizer_critic = torch.optim.AdamW(critic.parameters(), lr=experiment_config.training_config.lr_critic)
    
    scheduler = None
    if experiment_config.training_config.use_scheduler:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer_model,
            T_max=experiment_config.training_config.scheduler_T_max,
            eta_min=experiment_config.training_config.scheduler_eta_min
        )
    
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    mse = nn.MSELoss()
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, experiment_config.model_save_prefix + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    global_step = 0
    if experiment_config.checkpoint_path:
        try:
            starting_epoch, saved_metrics, global_step = load_checkpoint(
                experiment_config.checkpoint_path,
                model,
                critic,
                optimizer_model,
                optimizer_critic,
                scheduler,
                experiment_config,
                experiment_config.device
            )
            for param_group in optimizer_model.param_groups:
                param_group['lr'] = experiment_config.training_config.lr
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}, global_step {global_step}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0
            global_step = 0

    print("Starting training loop...")
    global_start = time.time()
    device = experiment_config.device

    # Training loop
    for epoch in range(starting_epoch, experiment_config.epochs):
        model.train()
        critic.train()
        if lm_ckpt_path:
            model.encoder.eval()
            
        epoch_loss = 0.0
        epoch_reconstruction_loss = 0.0
        epoch_avb_loss = 0.0
        epoch_critic_loss = 0.0
        epoch_gradient_penalty = 0.0
        epoch_p_score = 0.0
        epoch_q_score = 0.0
        epoch_acc = 0.0
        n_batches = 0
        start = time.time()

        # Training phase
        for batch, x in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            x = x.to(device)
            
            # x is [B, T] tensor of token trajectories. Virtually stack it into [B, T', C] windows
            B, T_data = x.shape
            C = model.context_window
            x_unfolded = x.unfold(dimension=-1, size=C, step=1)
            B, T, C_out = x_unfolded.shape
            x_reshaped = x_unfolded.reshape(B * T, C_out)
            
            # Get encoder outputs
            z, mu_q, logvar_q = model.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers)
            z_q_trajectories = z.reshape(B, T, model.latent_dim)

            # Store gradient penalty, q_scores, and p_scores from final critic training step
            final_gradient_penalty = 0
            final_q_score = 0
            final_p_score = 0
            final_critic_loss = 0

            # Critic Training
            for critic_step in range(experiment_config.critic_steps):
                # z_p_trajectories = model.generate_latent_trajectory(seq_len=T, device=device, z0=z_q_trajectories[:, 0, :])
                z_p_trajectories = model.generate_latent_trajectory(seq_len=T_generation, device=device, z0=z_q_trajectories[:, 0, :])
                
                q_scores = critic(z_q_trajectories[:, :T_generation].detach()).squeeze(-1)  # [B]
                p_scores = critic(z_p_trajectories.detach()).squeeze(-1)  # [B]

                gradient_penalty = compute_gradient_penalty(critic, z_q_trajectories[:, :T_generation].detach(), z_p_trajectories.detach(), device)
                loss_critic = p_scores.mean() - q_scores.mean() + experiment_config.training_config.gp_factor * gradient_penalty

                optimizer_critic.zero_grad()
                loss_critic.backward()
                optimizer_critic.step()
                
                # Store metrics from final critic step
                if critic_step == experiment_config.critic_steps - 1:
                    final_gradient_penalty = gradient_penalty.item()
                    final_q_score = q_scores.mean().item()
                    final_p_score = p_scores.mean().item()
                    final_critic_loss = loss_critic.item()

            ### VAE Training ###
            # Reconstruction
            logits = model.decode(z)
            loss_reconstruction = criterion(logits, x_reshaped[:, -1])
            acc = calculate_accuracy(logits, x_reshaped[:, -1], pad_id)

            # Generate prior trajectory and get critic scores
            # z_p_trajectories = model.generate_latent_trajectory(seq_len=T, device=device, z0=z_q_trajectories[:, 0, :])
            z_p_trajectories = model.generate_latent_trajectory(seq_len=T_generation, device=device, z0=z_q_trajectories[:, 0, :])
            q_scores = critic(z_q_trajectories[:, :T_generation]).squeeze(-1)
            p_scores = critic(z_p_trajectories).squeeze(-1)
            # print(f"p_scores: {p_scores[:, -1, -1].mean().item()}, q_scores: {q_scores[:, -1, -1].mean().item()}")

            # AVB loss
            avb_loss = -p_scores.mean() - q_scores.mean()
            # avb_loss = -p_scores.mean() + 1 * mse(q_scores, p_scores) # try this loss?

            # Total loss with annealing
            beta = (experiment_config.training_config.maximal_beta - experiment_config.training_config.minimal_beta) * \
                   min(1.0, global_step / experiment_config.training_config.warmup_steps) + \
                   experiment_config.training_config.minimal_beta
            loss = experiment_config.training_config.reconstruction_coef * loss_reconstruction + beta * avb_loss

            optimizer_model.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            if experiment_config.training_config.grad_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer_model.step()
            global_step += 1

            # Accumulate batch metrics
            epoch_loss += loss.item()
            epoch_reconstruction_loss += loss_reconstruction.item()
            epoch_avb_loss += avb_loss.item()
            epoch_critic_loss += final_critic_loss
            epoch_gradient_penalty += final_gradient_penalty
            epoch_p_score += final_p_score
            epoch_q_score += final_q_score
            epoch_acc += acc
            n_batches += 1

        # Validation phase
        val_loss, val_reconstruction_loss, val_avb_loss, val_critic_loss, val_p_score, val_q_score, val_acc = validate(
            model, critic, val_loader, criterion, device
        )

        # Calculate epoch averages
        train_loss = epoch_loss / n_batches
        train_reconstruction_loss = epoch_reconstruction_loss / n_batches
        train_avb_loss = epoch_avb_loss / n_batches
        train_critic_loss = epoch_critic_loss / n_batches
        train_gradient_penalty = epoch_gradient_penalty / n_batches
        train_p_score = epoch_p_score / n_batches
        train_q_score = epoch_q_score / n_batches
        train_acc = epoch_acc / n_batches

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_total_losses.append(train_loss)
        metrics.train_reconstruction_losses.append(train_reconstruction_loss)
        metrics.train_avb_losses.append(train_avb_loss)
        metrics.train_critic_losses.append(train_critic_loss)
        metrics.train_gradient_penalties.append(train_gradient_penalty)
        metrics.train_p_scores.append(train_p_score)
        metrics.train_q_scores.append(train_q_score)
        metrics.train_accuracies.append(train_acc)
        
        metrics.val_total_losses.append(val_loss)
        metrics.val_reconstruction_losses.append(val_reconstruction_loss)
        metrics.val_avb_losses.append(val_avb_loss)
        metrics.val_critic_losses.append(val_critic_loss)
        metrics.val_p_scores.append(val_p_score)
        metrics.val_q_scores.append(val_q_score)
        metrics.val_accuracies.append(val_acc)
        
        metrics.learning_rates.append(optimizer_model.param_groups[0]['lr'])

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Train Reconstruction Loss: {train_reconstruction_loss:.4f} | Train AVB Loss: {train_avb_loss:.4f}")
        print(f"Train Critic Loss: {train_critic_loss:.4f} | Train Gradient Penalty: {train_gradient_penalty:.4f}")
        print(f"Train P Score: {train_p_score:.4f} | Train Q Score: {train_q_score:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Val Reconstruction Loss: {val_reconstruction_loss:.4f} | Val AVB Loss: {val_avb_loss:.4f}")
        print(f"Val Critic Loss: {val_critic_loss:.4f}")
        print(f"Val P Score: {val_p_score:.4f} | Val Q Score: {val_q_score:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")
        
        # Step the scheduler
        if scheduler:
            scheduler.step()
            print(f"LR: {scheduler.get_last_lr()[0]:.6f}")

        # Save checkpoint
        if (epoch+1) % experiment_config.save_every == 0 or (epoch+1) == experiment_config.epochs:
            metrics.plot_metrics(save_dir)
            
            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            checkpoint_data = {
                'model_state_dict': model.state_dict(),
                'critic_state_dict': critic.state_dict(),
                'optimizer_model_state_dict': optimizer_model.state_dict(),
                'optimizer_critic_state_dict': optimizer_critic.state_dict(),
                'config': experiment_config,
                'epoch': epoch+1,
                'global_step': global_step,
                'metrics': {
                    'train_total_losses': metrics.train_total_losses,
                    'train_reconstruction_losses': metrics.train_reconstruction_losses,
                    'train_avb_losses': metrics.train_avb_losses,
                    'train_critic_losses': metrics.train_critic_losses,
                    'train_gradient_penalties': metrics.train_gradient_penalties,
                    'train_q_scores': metrics.train_q_scores,
                    'train_p_scores': metrics.train_p_scores,
                    'train_accuracies': metrics.train_accuracies,
                    'val_total_losses': metrics.val_total_losses,
                    'val_reconstruction_losses': metrics.val_reconstruction_losses,
                    'val_avb_losses': metrics.val_avb_losses,
                    'val_critic_losses': metrics.val_critic_losses,
                    'val_q_scores': metrics.val_q_scores,
                    'val_p_scores': metrics.val_p_scores,
                    'val_accuracies': metrics.val_accuracies,
                    'learning_rates': metrics.learning_rates
                }
            }
            if scheduler:
                checkpoint_data['scheduler_state_dict'] = scheduler.state_dict()
            
            torch.save(checkpoint_data, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    print("Training complete!")

In [ ]:
### Dynamical VAE Training - FIXED ENCODER TEACHER
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        # Training metrics
        self.train_total_losses = []
        self.train_reconstruction_losses = []
        self.train_kl_losses = []
        self.train_accuracies = []
        
        # Validation metrics
        self.val_reconstruction_losses = []
        self.val_kl_losses = []
        self.val_accuracies = []
        
        self.epochs = []
        self.learning_rates = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_total_losses = metrics_dict.get('train_total_losses', [])
        self.train_reconstruction_losses = metrics_dict.get('train_reconstruction_losses', [])
        self.train_kl_losses = metrics_dict.get('train_kl_losses', [])
        self.train_accuracies = metrics_dict.get('train_accuracies', [])
        
        self.val_reconstruction_losses = metrics_dict.get('val_reconstruction_losses', [])
        self.val_kl_losses = metrics_dict.get('val_kl_losses', [])
        self.val_accuracies = metrics_dict.get('val_accuracies', [])
        self.learning_rates = metrics_dict.get('learning_rates', [])
        
        self.epochs = list(range(1, len(self.train_total_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Color scheme
        train_color = "#287BBB"  # Blue
        val_color = "#ED9E00"    # Orange
        
        # Plot 1: Reconstruction losses
        axes[0,0].plot(self.epochs, self.train_reconstruction_losses, 
                    color=train_color, linestyle='-', label='Train Reconstruction')
        axes[0,0].plot(self.epochs, self.val_reconstruction_losses, 
                    color=val_color, linestyle='-', label='Val Reconstruction')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Loss')
        axes[0,0].set_title('Reconstruction Loss')
        axes[0,0].legend()
        axes[0,0].grid(True)

        # Plot 2: KL losses
        axes[0,1].plot(self.epochs, self.train_kl_losses, 
                    color=train_color, linestyle='-', label='Train KL')
        axes[0,1].plot(self.epochs, self.val_kl_losses, 
                    color=val_color, linestyle='-', label='Val KL')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Loss')
        axes[0,1].set_title('KL Divergence Loss')
        axes[0,1].legend()
        axes[0,1].grid(True)

        # Plot 3: Total training loss and LR
        ax3_twin = axes[1,0].twinx()
        axes[1,0].plot(self.epochs, self.train_total_losses, color=train_color, label='Total Train Loss')
        ax3_twin.plot(self.epochs, self.learning_rates, color='green', linestyle='--', label='Learning Rate')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Loss')
        ax3_twin.set_ylabel('Learning Rate')
        axes[1,0].set_title('Total Training Loss and Learning Rate')
        axes[1,0].legend(loc='upper left')
        ax3_twin.legend(loc='upper right')
        axes[1,0].grid(True)


        # Plot 4: Accuracies
        axes[1,1].plot(self.epochs, self.train_accuracies, 
                    color=train_color, linestyle='-', label='Train Accuracy')
        axes[1,1].plot(self.epochs, self.val_accuracies, 
                    color=val_color, linestyle='-', label='Val Accuracy')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Accuracy')
        axes[1,1].set_title('Model Accuracy')
        axes[1,1].legend()
        axes[1,1].grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png", dpi=300, bbox_inches='tight')

        plt.show()

def load_checkpoint(checkpoint_path, model, optimizer, scheduler, experiment_config, device):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch, loaded metrics, and global step
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler state if provided
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    # Compare relevant parts of the model config
    current_model_config = experiment_config.model_config
    saved_model_config = saved_config.model_config
    
    mismatched_keys = []
    for key in ['n_layers', 'n_heads', 'embed_dim', 'ffn_dim', 'context_window', 'latent_dim']:
        if getattr(current_model_config, key) != getattr(saved_model_config, key):
            mismatched_keys.append(key)
    
    if mismatched_keys:
        raise ValueError(f"Checkpoint config mismatch for keys: {mismatched_keys}")

    starting_epoch = checkpoint['epoch']
    global_step = checkpoint.get('global_step', 0)
    return starting_epoch, checkpoint['metrics'], global_step

n_trials = 1
for trial in range(n_trials):
    # Initialize model
    if experiment_config.model_config.mode == 'DVAE':
        model = TransformerDVAE(
            experiment_config.model_config
        ).to(experiment_config.device)

    else:
        raise ValueError(f"Unsupported mode: {experiment_config.model_config.mode}")

    # Load a pretrained LM model to use its encoder
    # Loading a saved model
    lm_ckpt_path = experiment_config.LM_checkpoint_path
    if lm_ckpt_path:
        LM_checkpoint = torch.load(lm_ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)

        if LM_checkpoint['config'].model_config.mode == 'LM':
            lm_model = TinyLlamaTransformer(
                LM_checkpoint['config'].model_config
            ).to(experiment_config.device)
        lm_model.load_state_dict(LM_checkpoint['model_state_dict'])
        model.encoder = lm_model
        for param in model.encoder.parameters():
            param.requires_grad = False
            
    model_fixed = TransformerDVAEFixedEncoder(
            experiment_config.model_config
        ).to(experiment_config.device)
    
    params_to_train = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = torch.optim.AdamW(params_to_train, lr=experiment_config.training_config.lr)
    scheduler = None
    if experiment_config.training_config.use_scheduler:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=experiment_config.training_config.scheduler_T_max,
            eta_min=experiment_config.training_config.scheduler_eta_min
        )
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    mse = nn.MSELoss()
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, experiment_config.model_save_prefix + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    global_step = 0
    if experiment_config.checkpoint_path:
        try:
            starting_epoch, saved_metrics, global_step = load_checkpoint(
                experiment_config.checkpoint_path,
                model,
                optimizer,
                scheduler,
                experiment_config,
                experiment_config.device
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = experiment_config.training_config.lr
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}, global_step {global_step}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0
            global_step = 0

    print("Starting training loop...")
    global_start = time.time()


    def validate(model: TransformerDVAE, val_loader, criterion, device):
        model.eval()
        total_reconstruction_loss = 0
        total_kl_loss = 0
        total_acc = 0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device)
                
                # x is [B, T] tensor of token trajectories. Virtually stack it into [B, T', C] windows
                B, T_data = x.shape
                C = model.context_window
                x_unfolded = x.unfold(dimension=-1, size=C, step=1)
                B, T, C_out = x_unfolded.shape
                
                x_reshaped = x_unfolded.reshape(B * T, C_out)
                
                # inference: encode observations into latents
                z, mu_q, logvar_q = model.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers) # z, mu, logvar: [B*T, latent_dim]
                z_fixed, mu_q_fixed, logvar_q_fixed = model_fixed.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers) # z, mu, logvar: [B*T, latent_dim]

                L_q = model.build_cholesky_L(logvar_q)

                # dist_q = MultivariateNormal(loc=mu_q, covariance_matrix=torch.diag_embed(torch.exp(logvar_q)))
                dist_q = MultivariateNormal(loc=mu_q, scale_tril=L_q)
                dist_p = MultivariateNormal(loc=mu_q_fixed, covariance_matrix=torch.diag_embed(torch.exp(logvar_q_fixed)))
                # dist_p = MultivariateNormal(loc=mu_p, scale_tril=L_p)

                KL_dist = kl_divergence(dist_q, dist_p)

                # Compute KL divergence loss
                # Dkl = kl_divergence_gaussians(mu_q, logvar_q, mu_p, logvar_p)  # [B*(T-1)]
                loss_kl = KL_dist.sum() / (B * (T-1) * model.latent_dim)

                # compute KL-annealing beta
                # beta = (experiment_config.training_config.maximal_beta - experiment_config.training_config.minimal_beta) * min(1.0, global_step / experiment_config.training_config.warmup_steps) + experiment_config.training_config.minimal_beta

                total_reconstruction_loss += 0
                total_kl_loss += loss_kl.item()
                total_acc += 0

        return total_reconstruction_loss / len(val_loader), total_kl_loss / len(val_loader), total_acc / len(val_loader)

    # Training loop
    device = experiment_config.device
    for epoch in range(starting_epoch, experiment_config.epochs):
        model.train()
        if lm_ckpt_path:
            model.encoder.eval()  # keep LM encoder in eval mode
        epoch_loss = 0.0
        epoch_loss_reconstruction = 0.0
        epoch_loss_kl = 0.0
        epoch_acc = 0.0
        start = time.time()

        # Training phase
        for batch, x in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            x = x.to(device)
            
            # x is [B, T] tensor of token trajectories. Virtually stack it into [B, T', C] windows
            B, T_data = x.shape
            C = model.context_window
            x_unfolded = x.unfold(dimension=-1, size=C, step=1)
            B, T, C_out = x_unfolded.shape
            
            x_reshaped = x_unfolded.reshape(B * T, C_out)
            
            # inference: encode observations into latents
            z, mu_q, logvar_q = model.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers) # z, mu, logvar: [B*T, latent_dim]
            z_fixed, mu_q_fixed, logvar_q_fixed = model_fixed.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers) # z, mu, logvar: [B*T, latent_dim]

            L_q = model.build_cholesky_L(logvar_q)

            # dist_q = MultivariateNormal(loc=mu_q, covariance_matrix=torch.diag_embed(torch.exp(logvar_q)))
            dist_q = MultivariateNormal(loc=mu_q, scale_tril=L_q)
            dist_p = MultivariateNormal(loc=mu_q_fixed, covariance_matrix=torch.diag_embed(torch.exp(logvar_q_fixed)))
            # dist_p = MultivariateNormal(loc=mu_p, scale_tril=L_p)

            KL_dist = kl_divergence(dist_q, dist_p)

            # Compute KL divergence loss
            # Dkl = kl_divergence_gaussians(mu_q, logvar_q, mu_p, logvar_p)  # [B*(T-1)]
            loss_kl = KL_dist.sum() / (B * (T-1) * model.latent_dim)

            # compute KL-annealing beta
            # beta = (experiment_config.training_config.maximal_beta - experiment_config.training_config.minimal_beta) * min(1.0, global_step / experiment_config.training_config.warmup_steps) + experiment_config.training_config.minimal_beta
            loss = loss_kl

            optimizer.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            if experiment_config.training_config.grad_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            global_step += 1

            epoch_loss += loss.item()
            epoch_loss_reconstruction += 0
            epoch_loss_kl += loss_kl.item()
            epoch_acc += 0

        # Validation phase
        val_reconstruction_loss, val_kl_loss, val_acc = validate(model, val_loader, criterion, device)

        # Calculate average losses and accuracies
        train_loss = epoch_loss / len(train_loader)
        train_acc = epoch_acc / len(train_loader)
        train_loss_reconstruction = epoch_loss_reconstruction / len(train_loader)
        train_loss_kl = epoch_loss_kl / len(train_loader)

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_total_losses.append(train_loss)
        metrics.train_reconstruction_losses.append(train_loss_reconstruction)
        metrics.train_kl_losses.append(train_loss_kl)
        metrics.train_accuracies.append(train_acc)
        metrics.val_reconstruction_losses.append(val_reconstruction_loss)
        metrics.val_kl_losses.append(val_kl_loss)
        metrics.val_accuracies.append(val_acc)
        metrics.learning_rates.append(optimizer.param_groups[0]['lr'])

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Total Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Train Reconstruction Loss: {train_loss_reconstruction:.4f} | Train KL Loss: {train_loss_kl:.4f}")
        print(f"Val Reconstruction Loss: {val_reconstruction_loss:.4f} | Val KL Loss: {val_kl_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")
        
        # Step the scheduler
        if scheduler:
            scheduler.step()
            print(f"LR: {scheduler.get_last_lr()[0]:.6f}")

        # Save checkpoint
        if (epoch+1) % experiment_config.save_every == 0 or (epoch+1) == experiment_config.epochs:
            # Plot metrics
            metrics.plot_metrics(save_dir)

            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}_FIXED_TEACHER.pt"
            checkpoint_data = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': experiment_config,
                'epoch': epoch+1,
                'global_step': global_step,
                'metrics': {
                    'train_total_losses': metrics.train_total_losses,
                    'train_reconstruction_losses': metrics.train_reconstruction_losses,
                    'train_kl_losses': metrics.train_kl_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'val_reconstruction_losses': metrics.val_reconstruction_losses,
                    'val_kl_losses': metrics.val_kl_losses,
                    'val_accuracies': metrics.val_accuracies,
                    'learning_rates': metrics.learning_rates
                }
            }
            if scheduler:
                checkpoint_data['scheduler_state_dict'] = scheduler.state_dict()
            
            torch.save(checkpoint_data, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")


    print("Training complete!")

In [ ]:
model.eval()
with torch.no_grad():
    z, mu, logvar = model.inference(
        torch.tensor(
            [[2,0,2,0,3,0,2,0,2,0,3,0,2,0,3,0,3,0,2,0,2,0,2,0,2,0,2,0,2,0,2,1],
             [2,0,2,0,3,0,2,0,2,0,3,0,2,0,3,0,3,0,2,0,2,0,2,0,2,0,2,0,2,0,3,1],
             [2,0,2,0,3,0,2,0,2,0,3,0,2,0,3,0,3,0,2,0,2,0,2,0,2,0,2,0,2,0,2,0],
             [2,0,2,0,3,0,2,0,2,0,3,0,2,0,3,0,3,0,2,0,2,0,2,0,2,0,2,0,2,0,3,0],
             [0,2,0,2,0,3,0,2,0,2,0,3,0,2,0,3,0,3,0,2,0,2,0,2,0,2,0,2,0,2,0,2],
             [0,2,0,2,0,3,0,2,0,2,0,3,0,2,0,3,0,3,0,2,0,2,0,2,0,2,0,2,0,2,0,3]], 
            dtype=torch.long, 
            device=experiment_config.device
        )
    )

In [ ]:
plt.figure()
plt.scatter(mu[:,0].detach().cpu().numpy(),mu[:,1].detach().cpu().numpy())
plt.grid()

In [ ]:
model.eval()
with torch.no_grad():
    enc_out = model(x_reshaped, n_layers=4, use_head=False)

In [ ]:
# last_token_activations = enc_out[:, -1:].reshape(x_reshaped[:].shape[0], -1) # [B, E]
# last_token_activations = enc_out[:, :].reshape(x_reshaped[:].shape[0], -1) # [B, E]
last_token_activations = enc_out[mask, -1:].reshape(x_reshaped[mask].shape[0], -1) # [B, E]
# last_token_activations = enc_out[mask * M1_mask, -1:].reshape(x_reshaped[mask * M1_mask].shape[0], -1) # [B, E]
# last_token_activations2 = enc_out[mask * M2_mask, -1:].reshape(x_reshaped[mask * M2_mask].shape[0], -1) # [B, E]

In [ ]:
plt.figure()
plt.plot(range(1, 33), pca.explained_variance_ratio_.cumsum(), '.')
plt.grid()

In [ ]:
pca.explained_variance_ratio_.cumsum()

In [ ]:
# LM_checkpoint = torch.load(lm_ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)
LM_checkpoint = torch.load("/home/galk/LanguageDynamics/models/linguistic_flip_flop/221025/tiny_LM_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_date_221025_1424_trial_1/ckpt_MANUAL.pt", map_location=torch.device(experiment_config.device), weights_only=False)
lm_model_checkpoint = TinyLlamaTransformer(
                LM_checkpoint['config'].model_config
            ).to(experiment_config.device)
lm_model_checkpoint.load_state_dict(LM_checkpoint['model_state_dict'])

In [ ]:
def compare_model_weights(model_1, model_2):
    """
    Compares the weights of two PyTorch models.

    Args:
        model_1 (nn.Module): The first PyTorch model.
        model_2 (nn.Module): The second PyTorch model.

    Returns:
        bool: True if all corresponding weights are identical, False otherwise.
    """
    models_differ = False
    for key_item_1, key_item_2 in zip(model_1.state_dict().items(), model_2.state_dict().items()):
        if key_item_1[0] != key_item_2[0]:
            print(f"Error: Mismatch in parameter names: {key_item_1[0]} vs {key_item_2[0]}")
            return False  # Or raise an exception if structure mismatch is critical

        if not torch.equal(key_item_1[1], key_item_2[1]):
            print(f"Mismatch found in parameter: {key_item_1[0]}")
            models_differ = True
            # Optionally, you can print the difference:
            # print(f"Difference: {key_item_1[1] - key_item_2[1]}")

    if not models_differ:
        print("Models match perfectly! :)")
        return True
    else:
        print("Models differ in weights.")
        return False
    
print(compare_model_weights(model, lm_model_checkpoint))

In [ ]:
plt.figure()
plt.scatter(z_q_trajectories[:,:T_generation,0].detach().cpu().numpy(),z_q_trajectories[:,:T_generation,1].detach().cpu().numpy())
plt.scatter(z_p_trajectories[:,:,0].detach().cpu().numpy(),z_p_trajectories[:,:,1].detach().cpu().numpy())

In [ ]:
critic.eval()
with torch.no_grad():
    q_scores = critic(z_q_trajectories[:, :T_generation]).squeeze(-1)
    p_scores = critic(z_p_trajectories).squeeze(-1)
    print(f"p_scores: {p_scores.mean().item()}, q_scores: {q_scores.mean().item()}")

In [ ]:
model